# Chaînes 10 — Le groupe des rotations comme automorphismes des directions dérivées

**Navigation** : [Index de la série](README.md) | [Version Python](Chaines-10-Groupe-Rotations-Automorphismes-Directions-Python.ipynb) | [Version C#](Chaines-10-Groupe-Rotations-Automorphismes-Directions-CSharp.ipynb)

Chapitre 49 du compendium *Chain Theory* (Wissner-Gross, 2026, preprint non relu, lu depuis la copie archivée au gisement), section 53. Deuxième chapitre de la Partie IV : la coordonnée d'ordre lue au chapitre précédent comme un compte, voici les rotations. Le patron de la série s'applique tel quel : **Mesuré** = ce que ce carnet énumère et vérifie lui-même (comptes de groupes finis, constantes de structure exactes, commutants résolus) ; **Cité** = la chaîne dérivative de la source (l'action sur les directions dérivées, le census des 102 bouts qubit) ; **Ouvert** = ce que la source laisse ouvert, ligne à ligne.

## Le cadre : après le compte, les rotations

« With order read as a count, the next chapter turns to the rotations. » Le chapitre 48 avait clos sur cette phrase. Le groupe des rotations n'est **pas posé** ici : il est *lu* comme les automorphismes des directions dérivées — le groupe que la littérature appelle SO(3) apparaît comme les symétries d'un ensemble fini de directions que la construction dérive elle-même. Trois objets finis portent le chapitre : le groupe d'identification au bout qubit (32 matrices monomiales à phases racines quatrièmes), son image sur les directions (le groupe diédral d'ordre 8), et le réseau des 24 rotations du cube où cette image s'incarne à indice trois.

## §53 — Théorème 53.1 (conditionnel) : le groupe des rotations

**Énoncé (traduit).** Le groupe des rotations de la théorie est le stabilisateur de l'unité hermitienne dans les isométries de la scission hermitienne de la trace — une algèbre de Lie de dimension trois, engendrée par les matrices *ad* des trois directions dérivées, aux crochets du symbole alterné ; son groupe connexe, les exponentielles aux points rationnels exacts, relevé au bout qubit par conjugaison par ses unitaires ; le relèvement fini, l'image diédrale d'ordre huit dans les vingt-quatre rotations du réseau. Chaque bout qubit des treize corps de la construction y est typé — proprement au bout proche d'un dock, improprement au bout loin d'un dock qubit-qubit —, les bouts qubit des docks sont spatiaux et les copies de hub internes.

**Standing : théorème conditionnel.** Prémisses : l'axiome, les six définitions du chapitre 1, la table Rosetta complète (chapitre I.1). Utilisé par : Open 17.1, Thm 54.1, Thm 57.1 et Open 57.2, Thm 59.1, Open 123.1, Thm 124.1.

In [2]:
// Bloc d'outillage commun : identique dans chaque cellule de code du carnet.
// evcxr conserve les items definis d'une cellule a l'autre ; un item de MEME nom
// redefini avec un contenu different casse les cellules voisines. On garde donc
// un prefixe canonique, repete mot pour mot, et chaque cellule n'ajoute que
// ses helpers propres.
use std::collections::BTreeMap;

#[derive(Clone, Copy, PartialEq, Eq, Debug)]
struct Fract { num: i128, den: i128 }

fn gcd(mut a: i128, mut b: i128) -> i128 {
    while b != 0 { let t = b; b = a % b; a = t; }
    a
}

impl Fract {
    fn new(num: i128, den: i128) -> Self {
        assert!(den != 0, "denominateur nul");
        let mut g = gcd(num.abs(), den.abs());
        if g == 0 { g = 1; }
        let mut n = num / g;
        let mut d = den / g;
        if d < 0 { n = -n; d = -d; }
        Fract { num: n, den: d }
    }
    fn zero() -> Self { Fract { num: 0, den: 1 } }
    fn add(&self, o: &Fract) -> Fract { Fract::new(self.num * o.den + o.num * self.den, self.den * o.den) }
    fn sub(&self, o: &Fract) -> Fract { Fract::new(self.num * o.den - o.num * self.den, self.den * o.den) }
    fn mul(&self, o: &Fract) -> Fract { Fract::new(self.num * o.num, self.den * o.den) }
    fn div(&self, o: &Fract) -> Fract { Fract::new(self.num * o.den, self.den * o.num) }
    fn neg(&self) -> Fract { Fract { num: -self.num, den: self.den } }
    fn is_zero(&self) -> bool { self.num == 0 }
}

impl std::fmt::Display for Fract {
    fn fmt(&self, f: &mut std::fmt::Formatter<'_>) -> std::fmt::Result {
        if self.den == 1 { write!(f, "{}", self.num) } else { write!(f, "{}/{}", self.num, self.den) }
    }
}

#[derive(Clone, Copy, PartialEq, Eq, Debug)]
struct C(Fract, Fract);

fn cmul(x: C, y: C) -> C { C(x.0.mul(&y.0).sub(&x.1.mul(&y.1)), x.0.mul(&y.1).add(&x.1.mul(&y.0))) }
fn cadd(x: C, y: C) -> C { C(x.0.add(&y.0), x.1.add(&y.1)) }
fn cneg(x: C) -> C { C(x.0.neg(), x.1.neg()) }
fn cscale(x: C, n: &Fract) -> C { C(x.0.mul(n), x.1.mul(n)) }

type MatC = Vec<Vec<C>>;
type Mat3 = [[i64; 3]; 3];
type MatN = Vec<Vec<i64>>;

fn mmul(a: &MatC, b: &MatC) -> MatC {
    let mut out = Vec::new();
    for i in 0..a.len() {
        let mut row = Vec::new();
        for j in 0..b[0].len() {
            let mut acc = C(Fract::zero(), Fract::zero());
            for k in 0..b.len() { acc = cadd(acc, cmul(a[i][k], b[k][j])); }
            row.push(acc);
        }
        out.push(row);
    }
    out
}

fn msub(a: &MatC, b: &MatC) -> MatC {
    let mut out = Vec::new();
    for i in 0..a.len() {
        let mut row = Vec::new();
        for j in 0..a[0].len() { row.push(cadd(a[i][j], cneg(b[i][j]))); }
        out.push(row);
    }
    out
}

fn crochet(a: &MatC, b: &MatC) -> MatC { msub(&mmul(a, b), &mmul(b, a)) }

fn meq(a: &MatC, b: &MatC) -> bool {
    (0..a.len()).all(|i| (0..a[0].len()).all(|j| a[i][j] == b[i][j]))
}

fn neg_mat(m: &MatC) -> MatC {
    m.iter().map(|row| row.iter().map(|x| cneg(*x)).collect()).collect()
}

fn mscale2(m: &MatC, k: i64) -> MatC {
    m.iter().map(|row| row.iter().map(|x| cscale(*x, &Fract::new(k as i128, 1))).collect()).collect()
}

fn det3(m: &Mat3) -> i64 {
    m[0][0] * (m[1][1] * m[2][2] - m[1][2] * m[2][1])
        - m[0][1] * (m[1][0] * m[2][2] - m[1][2] * m[2][0])
        + m[0][2] * (m[1][0] * m[2][1] - m[1][1] * m[2][0])
}

fn mul3(a: &Mat3, b: &Mat3) -> Mat3 {
    let mut out = [[0i64; 3]; 3];
    for i in 0..3 {
        for j in 0..3 {
            let mut acc = 0i64;
            for k in 0..3 { acc += a[i][k] * b[k][j]; }
            out[i][j] = acc;
        }
    }
    out
}

const I3: Mat3 = [[1, 0, 0], [0, 1, 0], [0, 0, 1]];

fn ordre3(m: &Mat3) -> i64 {
    let mut r = *m;
    let mut k = 1i64;
    while r != I3 { r = mul3(&r, m); k += 1; }
    k
}

fn py_bool(b: bool) -> String { if b { "True".to_string() } else { "False".to_string() } }

fn py_bools(items: &[bool]) -> String {
    format!("[{}]", items.iter().map(|b| py_bool(*b)).collect::<Vec<String>>().join(", "))
}

fn proportionnel(x: &MatC, base: &[MatC], c0: &C) -> Option<(usize, C)> {
    for (k, g) in base.iter().enumerate() {
        let mut c: Option<C> = None;
        let mut ok = true;
        for i in 0..2usize {
            for j in 0..2usize {
                let a = x[i][j];
                let b = g[i][j];
                if b == *c0 {
                    if a != *c0 { ok = false; }
                } else {
                    let den = b.0.mul(&b.0).add(&b.1.mul(&b.1));
                    let p = a.0.mul(&b.0).add(&a.1.mul(&b.1)).div(&den);
                    let q = a.1.mul(&b.0).sub(&a.0.mul(&b.1)).div(&den);
                    match c {
                        None => c = Some(C(p, q)),
                        Some(cc) => if cmul(cc, b) != a { ok = false; },
                    }
                }
            }
        }
        if ok { if let Some(cc) = c { return Some((k, cc)); } }
    }
    None
}

fn coeff_ad(t: &Vec<MatC>, k: usize, j: usize) -> Vec<i64> {
    let c = crochet(&t[k], &t[j]);
    let mut col = vec![0i64, 0, 0];
    for i in 0..3usize {
        if meq(&c, &t[i]) { col[i] = 1; }
        else if meq(&c, &neg_mat(&t[i])) { col[i] = -1; }
    }
    col
}

fn chi_a1(_g: &Mat3) -> i64 { 1 }
fn chi_a2(g: &Mat3) -> i64 { g[2][2] }
fn chi_e(g: &Mat3) -> i64 { g[0][0] + g[1][1] }
fn chi_dir(g: &Mat3) -> i64 { g[0][0] + g[1][1] + g[2][2] }
fn chi_pair(g: &Mat3) -> i64 { 1 + chi_dir(g) }

fn scalaire_interne(axe: &Vec<Mat3>, f: fn(&Mat3) -> i64, h: fn(&Mat3) -> i64) -> Fract {
    let somme: i128 = axe.iter().map(|g| (f(g) * h(g)) as i128).sum();
    Fract::new(somme, axe.len() as i128)
}

fn commutant(mats: &Vec<MatN>, n: usize) -> usize {
    let mut lignes: Vec<Vec<Fract>> = Vec::new();
    for g in mats.iter() {
        for i in 0..n {
            for j in 0..n {
                let mut ligne = vec![Fract::zero(); n * n];
                for k in 0..n {
                    ligne[n * i + k] = ligne[n * i + k].add(&Fract::new(g[k][j] as i128, 1));
                    ligne[n * k + j] = ligne[n * k + j].sub(&Fract::new(g[i][k] as i128, 1));
                }
                lignes.push(ligne);
            }
        }
    }
    let mut piv = 0usize;
    for col in 0..(n * n) {
        let sel = (piv..lignes.len()).find(|&r| !lignes[r][col].is_zero());
        let sel = match sel { Some(s) => s, None => continue };
        lignes.swap(piv, sel);
        let pv = lignes[piv][col];
        for v in lignes[piv].iter_mut() { *v = v.div(&pv); }
        for r in 0..lignes.len() {
            if r != piv && !lignes[r][col].is_zero() {
                let f = lignes[r][col];
                for c in 0..(n * n) {
                    lignes[r][c] = lignes[r][c].sub(&f.mul(&lignes[piv][c]));
                }
            }
        }
        piv += 1;
    }
    n * n - piv
}

// Arithmetique exacte dans Q(i) : chaque complexe est une paire de Fraction.


    let C0 = C(Fract::zero(), Fract::zero());
    let C1 = C(Fract::new(1, 1), Fract::zero());
    let CI = C(Fract::zero(), Fract::new(1, 1));

    // Le groupe d'identification au bout qubit, CITE comme « thirty-two monomial
    // matrices with fourth-root phases ». Nous l'enumerons : une matrice monomiale 2x2
    // = une colonne par ligne, une phase (1, i, -1, -i) par colonne.
    let racines = [C1, CI, cneg(C1), cneg(CI)];
    let mut monomiales: Vec<Vec<Vec<C>>> = Vec::new();
    for c0 in 0..2usize {
        for p1 in racines.iter() {
            for p2 in racines.iter() {
                let mut m = vec![vec![C0, C0], vec![C0, C0]];
                m[0][c0] = *p1;
                m[1][1 - c0] = *p2;
                monomiales.push(m);
            }
        }
    }
    // On ne compte que le noyau scalaire : evcxr ne persiste pas un Vec de
    // references a durée non statique.
    let n_scalaires = monomiales.iter()
        .filter(|m| m[0][1] == C0 && m[1][0] == C0 && m[0][0] == m[1][1])
        .count();
    println!("Matrices monomiales 2x2 a phases racines quatriemes : {} -> {}",
             monomiales.len(), py_bool(monomiales.len() == 32));
    println!("Matrices scalaires parmi elles (le noyau cite) : {} -> {}",
             n_scalaires, py_bool(n_scalaires == 4));
    let ratio = Fract::new(monomiales.len() as i128, n_scalaires as i128);
    println!("Arithmetique de Lagrange : {} / {} = {} ; l'image sur les directions est d'ordre 8 (CITÉ)",
             monomiales.len(), n_scalaires, ratio);


Matrices monomiales 2x2 a phases racines quatriemes : 32 -> True


Matrices scalaires parmi elles (le noyau cite) : 4 -> True


Arithmetique de Lagrange : 32 / 4 = 8 ; l'image sur les directions est d'ordre 8 (CITÉ)


### Lecture du résultat (§53, comptes)

**Mesuré** : les comptes du groupe fini — 32 matrices monomiales à phases racines quatrièmes (2 choix de colonne × 4² phases), dont exactement 4 scalaires (les phases ±1, ±i sur l'identité). **Cité** : l'application de ce groupe aux trois directions dérivées — sa définition appartient à la construction — dont l'image est le groupe diédral d'ordre huit, de noyau les quatre phases scalaires. L'arithmétique 32 = 4 × 8 est la cohérence de Lagrange entre les deux comptes ; elle ne prouve pas l'image, elle la contraint.

## Les vingt-quatre rotations du réseau

L'image diédrale vit « a subgroup of index three in the twenty-four lattice rotations ». Les vingt-quatre rotations du réseau sont les matrices de permutation signée 3×3 de déterminant +1 — le groupe des rotations du cube, dans la langue de la littérature. Nous les énumérons, puis nous réalisons le sous-groupe d'ordre huit comme le **stabilisateur d'un axe** : les rotations du cube qui préservent un axe à signe près. Cette réalisation est **nôtre** — la source n'écrit pas la matrice — mais elle porte exactement les étiquettes citées : les directions y portent le signe ⊕ le naturel.

In [3]:
// Bloc d'outillage commun : identique dans chaque cellule de code du carnet.
// evcxr conserve les items definis d'une cellule a l'autre ; un item de MEME nom
// redefini avec un contenu different casse les cellules voisines. On garde donc
// un prefixe canonique, repete mot pour mot, et chaque cellule n'ajoute que
// ses helpers propres.
use std::collections::BTreeMap;

#[derive(Clone, Copy, PartialEq, Eq, Debug)]
struct Fract { num: i128, den: i128 }

fn gcd(mut a: i128, mut b: i128) -> i128 {
    while b != 0 { let t = b; b = a % b; a = t; }
    a
}

impl Fract {
    fn new(num: i128, den: i128) -> Self {
        assert!(den != 0, "denominateur nul");
        let mut g = gcd(num.abs(), den.abs());
        if g == 0 { g = 1; }
        let mut n = num / g;
        let mut d = den / g;
        if d < 0 { n = -n; d = -d; }
        Fract { num: n, den: d }
    }
    fn zero() -> Self { Fract { num: 0, den: 1 } }
    fn add(&self, o: &Fract) -> Fract { Fract::new(self.num * o.den + o.num * self.den, self.den * o.den) }
    fn sub(&self, o: &Fract) -> Fract { Fract::new(self.num * o.den - o.num * self.den, self.den * o.den) }
    fn mul(&self, o: &Fract) -> Fract { Fract::new(self.num * o.num, self.den * o.den) }
    fn div(&self, o: &Fract) -> Fract { Fract::new(self.num * o.den, self.den * o.num) }
    fn neg(&self) -> Fract { Fract { num: -self.num, den: self.den } }
    fn is_zero(&self) -> bool { self.num == 0 }
}

impl std::fmt::Display for Fract {
    fn fmt(&self, f: &mut std::fmt::Formatter<'_>) -> std::fmt::Result {
        if self.den == 1 { write!(f, "{}", self.num) } else { write!(f, "{}/{}", self.num, self.den) }
    }
}

#[derive(Clone, Copy, PartialEq, Eq, Debug)]
struct C(Fract, Fract);

fn cmul(x: C, y: C) -> C { C(x.0.mul(&y.0).sub(&x.1.mul(&y.1)), x.0.mul(&y.1).add(&x.1.mul(&y.0))) }
fn cadd(x: C, y: C) -> C { C(x.0.add(&y.0), x.1.add(&y.1)) }
fn cneg(x: C) -> C { C(x.0.neg(), x.1.neg()) }
fn cscale(x: C, n: &Fract) -> C { C(x.0.mul(n), x.1.mul(n)) }

type MatC = Vec<Vec<C>>;
type Mat3 = [[i64; 3]; 3];
type MatN = Vec<Vec<i64>>;

fn mmul(a: &MatC, b: &MatC) -> MatC {
    let mut out = Vec::new();
    for i in 0..a.len() {
        let mut row = Vec::new();
        for j in 0..b[0].len() {
            let mut acc = C(Fract::zero(), Fract::zero());
            for k in 0..b.len() { acc = cadd(acc, cmul(a[i][k], b[k][j])); }
            row.push(acc);
        }
        out.push(row);
    }
    out
}

fn msub(a: &MatC, b: &MatC) -> MatC {
    let mut out = Vec::new();
    for i in 0..a.len() {
        let mut row = Vec::new();
        for j in 0..a[0].len() { row.push(cadd(a[i][j], cneg(b[i][j]))); }
        out.push(row);
    }
    out
}

fn crochet(a: &MatC, b: &MatC) -> MatC { msub(&mmul(a, b), &mmul(b, a)) }

fn meq(a: &MatC, b: &MatC) -> bool {
    (0..a.len()).all(|i| (0..a[0].len()).all(|j| a[i][j] == b[i][j]))
}

fn neg_mat(m: &MatC) -> MatC {
    m.iter().map(|row| row.iter().map(|x| cneg(*x)).collect()).collect()
}

fn mscale2(m: &MatC, k: i64) -> MatC {
    m.iter().map(|row| row.iter().map(|x| cscale(*x, &Fract::new(k as i128, 1))).collect()).collect()
}

fn det3(m: &Mat3) -> i64 {
    m[0][0] * (m[1][1] * m[2][2] - m[1][2] * m[2][1])
        - m[0][1] * (m[1][0] * m[2][2] - m[1][2] * m[2][0])
        + m[0][2] * (m[1][0] * m[2][1] - m[1][1] * m[2][0])
}

fn mul3(a: &Mat3, b: &Mat3) -> Mat3 {
    let mut out = [[0i64; 3]; 3];
    for i in 0..3 {
        for j in 0..3 {
            let mut acc = 0i64;
            for k in 0..3 { acc += a[i][k] * b[k][j]; }
            out[i][j] = acc;
        }
    }
    out
}

const I3: Mat3 = [[1, 0, 0], [0, 1, 0], [0, 0, 1]];

fn ordre3(m: &Mat3) -> i64 {
    let mut r = *m;
    let mut k = 1i64;
    while r != I3 { r = mul3(&r, m); k += 1; }
    k
}

fn py_bool(b: bool) -> String { if b { "True".to_string() } else { "False".to_string() } }

fn py_bools(items: &[bool]) -> String {
    format!("[{}]", items.iter().map(|b| py_bool(*b)).collect::<Vec<String>>().join(", "))
}

fn proportionnel(x: &MatC, base: &[MatC], c0: &C) -> Option<(usize, C)> {
    for (k, g) in base.iter().enumerate() {
        let mut c: Option<C> = None;
        let mut ok = true;
        for i in 0..2usize {
            for j in 0..2usize {
                let a = x[i][j];
                let b = g[i][j];
                if b == *c0 {
                    if a != *c0 { ok = false; }
                } else {
                    let den = b.0.mul(&b.0).add(&b.1.mul(&b.1));
                    let p = a.0.mul(&b.0).add(&a.1.mul(&b.1)).div(&den);
                    let q = a.1.mul(&b.0).sub(&a.0.mul(&b.1)).div(&den);
                    match c {
                        None => c = Some(C(p, q)),
                        Some(cc) => if cmul(cc, b) != a { ok = false; },
                    }
                }
            }
        }
        if ok { if let Some(cc) = c { return Some((k, cc)); } }
    }
    None
}

fn coeff_ad(t: &Vec<MatC>, k: usize, j: usize) -> Vec<i64> {
    let c = crochet(&t[k], &t[j]);
    let mut col = vec![0i64, 0, 0];
    for i in 0..3usize {
        if meq(&c, &t[i]) { col[i] = 1; }
        else if meq(&c, &neg_mat(&t[i])) { col[i] = -1; }
    }
    col
}

fn chi_a1(_g: &Mat3) -> i64 { 1 }
fn chi_a2(g: &Mat3) -> i64 { g[2][2] }
fn chi_e(g: &Mat3) -> i64 { g[0][0] + g[1][1] }
fn chi_dir(g: &Mat3) -> i64 { g[0][0] + g[1][1] + g[2][2] }
fn chi_pair(g: &Mat3) -> i64 { 1 + chi_dir(g) }

fn scalaire_interne(axe: &Vec<Mat3>, f: fn(&Mat3) -> i64, h: fn(&Mat3) -> i64) -> Fract {
    let somme: i128 = axe.iter().map(|g| (f(g) * h(g)) as i128).sum();
    Fract::new(somme, axe.len() as i128)
}

fn commutant(mats: &Vec<MatN>, n: usize) -> usize {
    let mut lignes: Vec<Vec<Fract>> = Vec::new();
    for g in mats.iter() {
        for i in 0..n {
            for j in 0..n {
                let mut ligne = vec![Fract::zero(); n * n];
                for k in 0..n {
                    ligne[n * i + k] = ligne[n * i + k].add(&Fract::new(g[k][j] as i128, 1));
                    ligne[n * k + j] = ligne[n * k + j].sub(&Fract::new(g[i][k] as i128, 1));
                }
                lignes.push(ligne);
            }
        }
    }
    let mut piv = 0usize;
    for col in 0..(n * n) {
        let sel = (piv..lignes.len()).find(|&r| !lignes[r][col].is_zero());
        let sel = match sel { Some(s) => s, None => continue };
        lignes.swap(piv, sel);
        let pv = lignes[piv][col];
        for v in lignes[piv].iter_mut() { *v = v.div(&pv); }
        for r in 0..lignes.len() {
            if r != piv && !lignes[r][col].is_zero() {
                let f = lignes[r][col];
                for c in 0..(n * n) {
                    lignes[r][c] = lignes[r][c].sub(&f.mul(&lignes[piv][c]));
                }
            }
        }
        piv += 1;
    }
    n * n - piv
}

// Les 24 rotations du reseau : permutations signees 3x3 de determinant +1.


    // Les six permutations de {0,1,2} en ordre lexicographique, puis les huit
    // triplets de signes dans l'ordre (1,1,1), (1,1,-1), ... : l'ordre du source.
    let perms: [[usize; 3]; 6] = [
        [0, 1, 2], [0, 2, 1], [1, 0, 2], [1, 2, 0], [2, 0, 1], [2, 1, 0],
    ];
    let signes: [[i64; 3]; 8] = [
        [1, 1, 1], [1, 1, -1], [1, -1, 1], [1, -1, -1],
        [-1, 1, 1], [-1, 1, -1], [-1, -1, 1], [-1, -1, -1],
    ];
    let mut rotations24: Vec<Mat3> = Vec::new();
    for perm in perms.iter() {
        for sg in signes.iter() {
            let mut m = [[0i64; 3]; 3];
            for i in 0..3 { m[i][perm[i]] = sg[i]; }
            if det3(&m) == 1 { rotations24.push(m); }
        }
    }
    println!("Permutations signees de determinant +1 : {} -> {}",
             rotations24.len(), py_bool(rotations24.len() == 24));

    // Notre realisation du sous-groupe d'ordre 8 : le stabilisateur de l'axe z.
    let axe: Vec<Mat3> = rotations24.iter()
        .filter(|m| m[0][2] == 0 && m[1][2] == 0 && m[2][0] == 0 && m[2][1] == 0)
        .cloned()
        .collect();
    let mut profil: Vec<i64> = axe.iter().map(ordre3).collect();
    profil.sort_unstable();
    let ferme = axe.iter().all(|a| axe.iter().all(|b| axe.contains(&mul3(a, b))));
    println!("Stabilisateur de l'axe (ordre) : {} -> {} ; stable par produit : {}",
             axe.len(), py_bool(axe.len() == 8), py_bool(ferme));
    println!("Profil par ordres : {:?} (D8 : cinq elements d'ordre 2, deux d'ordre 4 ; Q8 en aurait six d'ordre 4)", profil);
    let indice = Fract::new(rotations24.len() as i128, axe.len() as i128);
    println!("Indice dans les 24 : 24 / 8 = {} (indice trois, CITÉ)", indice);


Permutations signees de determinant +1 : 24 -> True


Stabilisateur de l'axe (ordre) : 8 -> True ; stable par produit : True


Profil par ordres : [1, 2, 2, 2, 2, 2, 4, 4] (D8 : cinq elements d'ordre 2, deux d'ordre 4 ; Q8 en aurait six d'ordre 4)


Indice dans les 24 : 24 / 8 = 3 (indice trois, CITÉ)


In [4]:
// Bloc d'outillage commun : identique dans chaque cellule de code du carnet.
// evcxr conserve les items definis d'une cellule a l'autre ; un item de MEME nom
// redefini avec un contenu different casse les cellules voisines. On garde donc
// un prefixe canonique, repete mot pour mot, et chaque cellule n'ajoute que
// ses helpers propres.
use std::collections::BTreeMap;

#[derive(Clone, Copy, PartialEq, Eq, Debug)]
struct Fract { num: i128, den: i128 }

fn gcd(mut a: i128, mut b: i128) -> i128 {
    while b != 0 { let t = b; b = a % b; a = t; }
    a
}

impl Fract {
    fn new(num: i128, den: i128) -> Self {
        assert!(den != 0, "denominateur nul");
        let mut g = gcd(num.abs(), den.abs());
        if g == 0 { g = 1; }
        let mut n = num / g;
        let mut d = den / g;
        if d < 0 { n = -n; d = -d; }
        Fract { num: n, den: d }
    }
    fn zero() -> Self { Fract { num: 0, den: 1 } }
    fn add(&self, o: &Fract) -> Fract { Fract::new(self.num * o.den + o.num * self.den, self.den * o.den) }
    fn sub(&self, o: &Fract) -> Fract { Fract::new(self.num * o.den - o.num * self.den, self.den * o.den) }
    fn mul(&self, o: &Fract) -> Fract { Fract::new(self.num * o.num, self.den * o.den) }
    fn div(&self, o: &Fract) -> Fract { Fract::new(self.num * o.den, self.den * o.num) }
    fn neg(&self) -> Fract { Fract { num: -self.num, den: self.den } }
    fn is_zero(&self) -> bool { self.num == 0 }
}

impl std::fmt::Display for Fract {
    fn fmt(&self, f: &mut std::fmt::Formatter<'_>) -> std::fmt::Result {
        if self.den == 1 { write!(f, "{}", self.num) } else { write!(f, "{}/{}", self.num, self.den) }
    }
}

#[derive(Clone, Copy, PartialEq, Eq, Debug)]
struct C(Fract, Fract);

fn cmul(x: C, y: C) -> C { C(x.0.mul(&y.0).sub(&x.1.mul(&y.1)), x.0.mul(&y.1).add(&x.1.mul(&y.0))) }
fn cadd(x: C, y: C) -> C { C(x.0.add(&y.0), x.1.add(&y.1)) }
fn cneg(x: C) -> C { C(x.0.neg(), x.1.neg()) }
fn cscale(x: C, n: &Fract) -> C { C(x.0.mul(n), x.1.mul(n)) }

type MatC = Vec<Vec<C>>;
type Mat3 = [[i64; 3]; 3];
type MatN = Vec<Vec<i64>>;

fn mmul(a: &MatC, b: &MatC) -> MatC {
    let mut out = Vec::new();
    for i in 0..a.len() {
        let mut row = Vec::new();
        for j in 0..b[0].len() {
            let mut acc = C(Fract::zero(), Fract::zero());
            for k in 0..b.len() { acc = cadd(acc, cmul(a[i][k], b[k][j])); }
            row.push(acc);
        }
        out.push(row);
    }
    out
}

fn msub(a: &MatC, b: &MatC) -> MatC {
    let mut out = Vec::new();
    for i in 0..a.len() {
        let mut row = Vec::new();
        for j in 0..a[0].len() { row.push(cadd(a[i][j], cneg(b[i][j]))); }
        out.push(row);
    }
    out
}

fn crochet(a: &MatC, b: &MatC) -> MatC { msub(&mmul(a, b), &mmul(b, a)) }

fn meq(a: &MatC, b: &MatC) -> bool {
    (0..a.len()).all(|i| (0..a[0].len()).all(|j| a[i][j] == b[i][j]))
}

fn neg_mat(m: &MatC) -> MatC {
    m.iter().map(|row| row.iter().map(|x| cneg(*x)).collect()).collect()
}

fn mscale2(m: &MatC, k: i64) -> MatC {
    m.iter().map(|row| row.iter().map(|x| cscale(*x, &Fract::new(k as i128, 1))).collect()).collect()
}

fn det3(m: &Mat3) -> i64 {
    m[0][0] * (m[1][1] * m[2][2] - m[1][2] * m[2][1])
        - m[0][1] * (m[1][0] * m[2][2] - m[1][2] * m[2][0])
        + m[0][2] * (m[1][0] * m[2][1] - m[1][1] * m[2][0])
}

fn mul3(a: &Mat3, b: &Mat3) -> Mat3 {
    let mut out = [[0i64; 3]; 3];
    for i in 0..3 {
        for j in 0..3 {
            let mut acc = 0i64;
            for k in 0..3 { acc += a[i][k] * b[k][j]; }
            out[i][j] = acc;
        }
    }
    out
}

const I3: Mat3 = [[1, 0, 0], [0, 1, 0], [0, 0, 1]];

fn ordre3(m: &Mat3) -> i64 {
    let mut r = *m;
    let mut k = 1i64;
    while r != I3 { r = mul3(&r, m); k += 1; }
    k
}

fn py_bool(b: bool) -> String { if b { "True".to_string() } else { "False".to_string() } }

fn py_bools(items: &[bool]) -> String {
    format!("[{}]", items.iter().map(|b| py_bool(*b)).collect::<Vec<String>>().join(", "))
}

fn proportionnel(x: &MatC, base: &[MatC], c0: &C) -> Option<(usize, C)> {
    for (k, g) in base.iter().enumerate() {
        let mut c: Option<C> = None;
        let mut ok = true;
        for i in 0..2usize {
            for j in 0..2usize {
                let a = x[i][j];
                let b = g[i][j];
                if b == *c0 {
                    if a != *c0 { ok = false; }
                } else {
                    let den = b.0.mul(&b.0).add(&b.1.mul(&b.1));
                    let p = a.0.mul(&b.0).add(&a.1.mul(&b.1)).div(&den);
                    let q = a.1.mul(&b.0).sub(&a.0.mul(&b.1)).div(&den);
                    match c {
                        None => c = Some(C(p, q)),
                        Some(cc) => if cmul(cc, b) != a { ok = false; },
                    }
                }
            }
        }
        if ok { if let Some(cc) = c { return Some((k, cc)); } }
    }
    None
}

fn coeff_ad(t: &Vec<MatC>, k: usize, j: usize) -> Vec<i64> {
    let c = crochet(&t[k], &t[j]);
    let mut col = vec![0i64, 0, 0];
    for i in 0..3usize {
        if meq(&c, &t[i]) { col[i] = 1; }
        else if meq(&c, &neg_mat(&t[i])) { col[i] = -1; }
    }
    col
}

fn chi_a1(_g: &Mat3) -> i64 { 1 }
fn chi_a2(g: &Mat3) -> i64 { g[2][2] }
fn chi_e(g: &Mat3) -> i64 { g[0][0] + g[1][1] }
fn chi_dir(g: &Mat3) -> i64 { g[0][0] + g[1][1] + g[2][2] }
fn chi_pair(g: &Mat3) -> i64 { 1 + chi_dir(g) }

fn scalaire_interne(axe: &Vec<Mat3>, f: fn(&Mat3) -> i64, h: fn(&Mat3) -> i64) -> Fract {
    let somme: i128 = axe.iter().map(|g| (f(g) * h(g)) as i128).sum();
    Fract::new(somme, axe.len() as i128)
}

fn commutant(mats: &Vec<MatN>, n: usize) -> usize {
    let mut lignes: Vec<Vec<Fract>> = Vec::new();
    for g in mats.iter() {
        for i in 0..n {
            for j in 0..n {
                let mut ligne = vec![Fract::zero(); n * n];
                for k in 0..n {
                    ligne[n * i + k] = ligne[n * i + k].add(&Fract::new(g[k][j] as i128, 1));
                    ligne[n * k + j] = ligne[n * k + j].sub(&Fract::new(g[i][k] as i128, 1));
                }
                lignes.push(ligne);
            }
        }
    }
    let mut piv = 0usize;
    for col in 0..(n * n) {
        let sel = (piv..lignes.len()).find(|&r| !lignes[r][col].is_zero());
        let sel = match sel { Some(s) => s, None => continue };
        lignes.swap(piv, sel);
        let pv = lignes[piv][col];
        for v in lignes[piv].iter_mut() { *v = v.div(&pv); }
        for r in 0..lignes.len() {
            if r != piv && !lignes[r][col].is_zero() {
                let f = lignes[r][col];
                for c in 0..(n * n) {
                    lignes[r][c] = lignes[r][c].sub(&f.mul(&lignes[piv][c]));
                }
            }
        }
        piv += 1;
    }
    n * n - piv
}

// La signature D8 se lit aussi sur les classes de conjugaison du stabilisateur :
// l'element neutre, le demi-tour central (trace -2 au bloc xy), les deux quarts
// de tour (ordre 4), et les QUATRE reflexions -- diagonales et hors-diagonales,
// deux classes distinctes de deux. Cinq classes = cinq irreductibles, les
// etiquettes A1, A2, B1, B2, E de la litterature [citees].

fn bloc_xy(g: &Mat3) -> [[i64; 2]; 2] { [[g[0][0], g[0][1]], [g[1][0], g[1][1]]] }


    let perms: [[usize; 3]; 6] = [
        [0, 1, 2], [0, 2, 1], [1, 0, 2], [1, 2, 0], [2, 0, 1], [2, 1, 0],
    ];
    let signes: [[i64; 3]; 8] = [
        [1, 1, 1], [1, 1, -1], [1, -1, 1], [1, -1, -1],
        [-1, 1, 1], [-1, 1, -1], [-1, -1, 1], [-1, -1, -1],
    ];
    let mut rotations24: Vec<Mat3> = Vec::new();
    for perm in perms.iter() {
        for sg in signes.iter() {
            let mut m = [[0i64; 3]; 3];
            for i in 0..3 { m[i][perm[i]] = sg[i]; }
            if det3(&m) == 1 { rotations24.push(m); }
        }
    }
    let axe: Vec<Mat3> = rotations24.iter()
        .filter(|m| m[0][2] == 0 && m[1][2] == 0 && m[2][0] == 0 && m[2][1] == 0)
        .cloned()
        .collect();

    // Le bloc xy : les deux premieres lignes et colonnes.
    // La cle d'une classe : (ordre, trace du bloc xy, hors-diagonale ?) -- l'ordre
    // BTreeMap reproduit le tri du source, False < True.
    let mut classes: BTreeMap<(i64, i64, bool), usize> = BTreeMap::new();
    for g in axe.iter() {
        let b = bloc_xy(g);
        let tr = b[0][0] + b[1][1];
        let hors_diag = b[0][1] != 0;
        *classes.entry((ordre3(g), tr, hors_diag)).or_insert(0) += 1;
    }
    for (cle, n) in classes.iter() {
        let etiquette = if cle.2 { "hors-diagonale" } else { "diagonale" };
        println!("ordre {} | trace bloc xy {} | {} -> {} element(s)", cle.0, cle.1, etiquette, n);
    }
    println!("Classes distingues : {} -> {} (une par irreductible : A1, A2, B1, B2, E, citées)",
             classes.len(), py_bool(classes.len() == 5));


ordre 1 | trace bloc xy 2 | diagonale -> 1 element(s)


ordre 2 | trace bloc xy -2 | diagonale -> 1 element(s)


ordre 2 | trace bloc xy 0 | diagonale -> 2 element(s)


ordre 2 | trace bloc xy 0 | hors-diagonale -> 2 element(s)


ordre 4 | trace bloc xy 0 | hors-diagonale -> 2 element(s)


Classes distingues : 5 -> True (une par irreductible : A1, A2, B1, B2, E, citées)


### Lecture du résultat (réseau)

**Mesuré** : 24 permutations signées de déterminant +1 ; le stabilisateur d'axe y forme un groupe d'ordre 8, fermé par produit, au profil d'ordres [1, 2, 2, 2, 2, 2, 4, 4] — la signature du diédral D8, que le groupe des quaternions Q8 n'a pas (six éléments d'ordre 4) — et à cinq classes de conjugaison. **Cité** : l'indice trois de l'image dans les vingt-quatre, et l'appartenance de cette réalisation à la construction de la source (la nôtre en est une incarnation, pas la définition).

## L'algèbre de Lie de dimension trois

« a Lie algebra of dimension three spanned by the ad matrices of the three derived directions with the brackets of the alternating symbol » — l'algèbre des rotations est engendrée par les trois directions dérivées, aux crochets $[T_a, T_b] = \epsilon_{abc} T_c$. Le compendium imprime le triple transposé au chapitre suivant (p. 147, dérivation du Thm 55.1) : $\left(\begin{smallmatrix}0 & -i/2\\ -i/2 & 0\end{smallmatrix}\right)$, $\left(\begin{smallmatrix}0 & 1/2\\ -1/2 & 0\end{smallmatrix}\right)$, $\left(\begin{smallmatrix}-i/2 & 0\\ 0 & i/2\end{smallmatrix}\right)$ — à un facteur près, les matrices de Pauli. Nous en mesurons les constantes de structure exactes dans $\mathbb{Q}(i)$, puis la ré-orientation d'un signe qui les ramène au symbole alterné — le compendium dit lui-même, au ch. 55, que la base littérale portée demande une ré-orientation pour retrouver la table des crochets.

In [5]:
// Bloc d'outillage commun : identique dans chaque cellule de code du carnet.
// evcxr conserve les items definis d'une cellule a l'autre ; un item de MEME nom
// redefini avec un contenu different casse les cellules voisines. On garde donc
// un prefixe canonique, repete mot pour mot, et chaque cellule n'ajoute que
// ses helpers propres.
use std::collections::BTreeMap;

#[derive(Clone, Copy, PartialEq, Eq, Debug)]
struct Fract { num: i128, den: i128 }

fn gcd(mut a: i128, mut b: i128) -> i128 {
    while b != 0 { let t = b; b = a % b; a = t; }
    a
}

impl Fract {
    fn new(num: i128, den: i128) -> Self {
        assert!(den != 0, "denominateur nul");
        let mut g = gcd(num.abs(), den.abs());
        if g == 0 { g = 1; }
        let mut n = num / g;
        let mut d = den / g;
        if d < 0 { n = -n; d = -d; }
        Fract { num: n, den: d }
    }
    fn zero() -> Self { Fract { num: 0, den: 1 } }
    fn add(&self, o: &Fract) -> Fract { Fract::new(self.num * o.den + o.num * self.den, self.den * o.den) }
    fn sub(&self, o: &Fract) -> Fract { Fract::new(self.num * o.den - o.num * self.den, self.den * o.den) }
    fn mul(&self, o: &Fract) -> Fract { Fract::new(self.num * o.num, self.den * o.den) }
    fn div(&self, o: &Fract) -> Fract { Fract::new(self.num * o.den, self.den * o.num) }
    fn neg(&self) -> Fract { Fract { num: -self.num, den: self.den } }
    fn is_zero(&self) -> bool { self.num == 0 }
}

impl std::fmt::Display for Fract {
    fn fmt(&self, f: &mut std::fmt::Formatter<'_>) -> std::fmt::Result {
        if self.den == 1 { write!(f, "{}", self.num) } else { write!(f, "{}/{}", self.num, self.den) }
    }
}

#[derive(Clone, Copy, PartialEq, Eq, Debug)]
struct C(Fract, Fract);

fn cmul(x: C, y: C) -> C { C(x.0.mul(&y.0).sub(&x.1.mul(&y.1)), x.0.mul(&y.1).add(&x.1.mul(&y.0))) }
fn cadd(x: C, y: C) -> C { C(x.0.add(&y.0), x.1.add(&y.1)) }
fn cneg(x: C) -> C { C(x.0.neg(), x.1.neg()) }
fn cscale(x: C, n: &Fract) -> C { C(x.0.mul(n), x.1.mul(n)) }

type MatC = Vec<Vec<C>>;
type Mat3 = [[i64; 3]; 3];
type MatN = Vec<Vec<i64>>;

fn mmul(a: &MatC, b: &MatC) -> MatC {
    let mut out = Vec::new();
    for i in 0..a.len() {
        let mut row = Vec::new();
        for j in 0..b[0].len() {
            let mut acc = C(Fract::zero(), Fract::zero());
            for k in 0..b.len() { acc = cadd(acc, cmul(a[i][k], b[k][j])); }
            row.push(acc);
        }
        out.push(row);
    }
    out
}

fn msub(a: &MatC, b: &MatC) -> MatC {
    let mut out = Vec::new();
    for i in 0..a.len() {
        let mut row = Vec::new();
        for j in 0..a[0].len() { row.push(cadd(a[i][j], cneg(b[i][j]))); }
        out.push(row);
    }
    out
}

fn crochet(a: &MatC, b: &MatC) -> MatC { msub(&mmul(a, b), &mmul(b, a)) }

fn meq(a: &MatC, b: &MatC) -> bool {
    (0..a.len()).all(|i| (0..a[0].len()).all(|j| a[i][j] == b[i][j]))
}

fn neg_mat(m: &MatC) -> MatC {
    m.iter().map(|row| row.iter().map(|x| cneg(*x)).collect()).collect()
}

fn mscale2(m: &MatC, k: i64) -> MatC {
    m.iter().map(|row| row.iter().map(|x| cscale(*x, &Fract::new(k as i128, 1))).collect()).collect()
}

fn det3(m: &Mat3) -> i64 {
    m[0][0] * (m[1][1] * m[2][2] - m[1][2] * m[2][1])
        - m[0][1] * (m[1][0] * m[2][2] - m[1][2] * m[2][0])
        + m[0][2] * (m[1][0] * m[2][1] - m[1][1] * m[2][0])
}

fn mul3(a: &Mat3, b: &Mat3) -> Mat3 {
    let mut out = [[0i64; 3]; 3];
    for i in 0..3 {
        for j in 0..3 {
            let mut acc = 0i64;
            for k in 0..3 { acc += a[i][k] * b[k][j]; }
            out[i][j] = acc;
        }
    }
    out
}

const I3: Mat3 = [[1, 0, 0], [0, 1, 0], [0, 0, 1]];

fn ordre3(m: &Mat3) -> i64 {
    let mut r = *m;
    let mut k = 1i64;
    while r != I3 { r = mul3(&r, m); k += 1; }
    k
}

fn py_bool(b: bool) -> String { if b { "True".to_string() } else { "False".to_string() } }

fn py_bools(items: &[bool]) -> String {
    format!("[{}]", items.iter().map(|b| py_bool(*b)).collect::<Vec<String>>().join(", "))
}

fn proportionnel(x: &MatC, base: &[MatC], c0: &C) -> Option<(usize, C)> {
    for (k, g) in base.iter().enumerate() {
        let mut c: Option<C> = None;
        let mut ok = true;
        for i in 0..2usize {
            for j in 0..2usize {
                let a = x[i][j];
                let b = g[i][j];
                if b == *c0 {
                    if a != *c0 { ok = false; }
                } else {
                    let den = b.0.mul(&b.0).add(&b.1.mul(&b.1));
                    let p = a.0.mul(&b.0).add(&a.1.mul(&b.1)).div(&den);
                    let q = a.1.mul(&b.0).sub(&a.0.mul(&b.1)).div(&den);
                    match c {
                        None => c = Some(C(p, q)),
                        Some(cc) => if cmul(cc, b) != a { ok = false; },
                    }
                }
            }
        }
        if ok { if let Some(cc) = c { return Some((k, cc)); } }
    }
    None
}

fn coeff_ad(t: &Vec<MatC>, k: usize, j: usize) -> Vec<i64> {
    let c = crochet(&t[k], &t[j]);
    let mut col = vec![0i64, 0, 0];
    for i in 0..3usize {
        if meq(&c, &t[i]) { col[i] = 1; }
        else if meq(&c, &neg_mat(&t[i])) { col[i] = -1; }
    }
    col
}

fn chi_a1(_g: &Mat3) -> i64 { 1 }
fn chi_a2(g: &Mat3) -> i64 { g[2][2] }
fn chi_e(g: &Mat3) -> i64 { g[0][0] + g[1][1] }
fn chi_dir(g: &Mat3) -> i64 { g[0][0] + g[1][1] + g[2][2] }
fn chi_pair(g: &Mat3) -> i64 { 1 + chi_dir(g) }

fn scalaire_interne(axe: &Vec<Mat3>, f: fn(&Mat3) -> i64, h: fn(&Mat3) -> i64) -> Fract {
    let somme: i128 = axe.iter().map(|g| (f(g) * h(g)) as i128).sum();
    Fract::new(somme, axe.len() as i128)
}

fn commutant(mats: &Vec<MatN>, n: usize) -> usize {
    let mut lignes: Vec<Vec<Fract>> = Vec::new();
    for g in mats.iter() {
        for i in 0..n {
            for j in 0..n {
                let mut ligne = vec![Fract::zero(); n * n];
                for k in 0..n {
                    ligne[n * i + k] = ligne[n * i + k].add(&Fract::new(g[k][j] as i128, 1));
                    ligne[n * k + j] = ligne[n * k + j].sub(&Fract::new(g[i][k] as i128, 1));
                }
                lignes.push(ligne);
            }
        }
    }
    let mut piv = 0usize;
    for col in 0..(n * n) {
        let sel = (piv..lignes.len()).find(|&r| !lignes[r][col].is_zero());
        let sel = match sel { Some(s) => s, None => continue };
        lignes.swap(piv, sel);
        let pv = lignes[piv][col];
        for v in lignes[piv].iter_mut() { *v = v.div(&pv); }
        for r in 0..lignes.len() {
            if r != piv && !lignes[r][col].is_zero() {
                let f = lignes[r][col];
                for c in 0..(n * n) {
                    lignes[r][c] = lignes[r][c].sub(&f.mul(&lignes[piv][c]));
                }
            }
        }
        piv += 1;
    }
    n * n - piv
}


    let C0 = C(Fract::zero(), Fract::zero());

    // Le triple littéral du compendium (ch. 55, p. 147, cité).
    let demi = Fract::new(1, 2);
    let g1 = vec![vec![C0, cscale(cneg(C(Fract::zero(), Fract::new(1, 1))), &demi)],
                   vec![cscale(cneg(C(Fract::zero(), Fract::new(1, 1))), &demi), C0]];
    let g2 = vec![vec![C0, C(demi, Fract::zero())], vec![cneg(C(demi, Fract::zero())), C0]];
    let g3 = vec![vec![cscale(cneg(C(Fract::zero(), Fract::new(1, 1))), &demi), C0],
                   vec![C0, cscale(C(Fract::zero(), Fract::new(1, 1)), &demi)]];
    let litteral: Vec<MatC> = vec![g1, g2, g3];

    // proportionnel : cherche la base g telle que x = c * g, coefficient complexe.
    // (fonction canonique : cf. le bloc d'outillage commun)

    println!("Constantes de structure du triple littéral :");
    let mut constantes: Vec<Fract> = Vec::new();
    for (a, b) in [(0usize, 1usize), (1, 2), (2, 0)] {
        let cro = crochet(&litteral[a], &litteral[b]);
        let (k, c) = match proportionnel(&cro, &litteral, &C0) {
            Some(v) => v,
            None => { println!("constante introuvable"); (0, C0) }
        };
        constantes.push(c.0);
        println!("  [G{},G{}] = {} * G{}", a + 1, b + 1, c.0, k + 1);
    }
    let litteral_ok = constantes.iter().all(|c| *c == Fract::new(-1, 1));
    println!("Les trois constantes au signe moins : {}", py_bool(litteral_ok));

    // Re-orientation : un seul signe retourne (le deuxieme generateur) et les crochets
    // deviennent exactement ceux du symbole alterne.
    let t: Vec<MatC> = vec![litteral[0].clone(), neg_mat(&litteral[1]), litteral[2].clone()];
    let attendu: [(usize, usize, usize); 3] = [(0, 1, 2), (1, 2, 0), (2, 0, 1)];
    let epsilon_exact = attendu.iter().all(|&(a, b, k)| meq(&crochet(&t[a], &t[b]), &t[k]));
    println!("Re-orienté (flip de G2) : [Ta,Tb] = eps_abl*Tc exact -> {}", py_bool(epsilon_exact));
    let moins_quart_i = vec![vec![C(Fract::new(-1, 4), Fract::zero()), C0],
                             vec![C0, C(Fract::new(-1, 4), Fract::zero())]];
    let carres: Vec<bool> = t.iter().map(|x| meq(&mmul(x, x), &moins_quart_i)).collect();
    println!("Tj^2 = -(1/4)*I pour j = 1, 2, 3 : {} -> {}", py_bools(&carres), py_bool(carres.iter().all(|b| *b)));


Constantes de structure du triple littéral :


  [G1,G2] = -1 * G3


  [G2,G3] = -1 * G1


  [G3,G1] = -1 * G2


Les trois constantes au signe moins : True


Re-orienté (flip de G2) : [Ta,Tb] = eps_abl*Tc exact -> True


Tj^2 = -(1/4)*I pour j = 1, 2, 3 : [True, True, True] -> True


### Les exponentielles aux points rationnels et le double recouvrement

Puisque $T_j^2 = -\tfrac{1}{4} I$, l'exponentielle a la forme fermée $\exp(\theta T_j) = \cos(\theta/2) I + 2\sin(\theta/2) T_j$. Aux angles rationnels en demi-tours, tout est exact dans $\mathbb{Q}(i)$ : le demi-tour $\theta = \pi$ donne $2T_3 = -i\sigma_z$ ; le tour complet $\theta = 2\pi$ donne $-I$ **au bout qubit** — pendant que l'action conjuguée sur les directions est l'identité. C'est le double recouvrement de la littérature, mesuré : « the full rotation is minus one at every qubit end and plus one on the directions ».

In [6]:
// Bloc d'outillage commun : identique dans chaque cellule de code du carnet.
// evcxr conserve les items definis d'une cellule a l'autre ; un item de MEME nom
// redefini avec un contenu different casse les cellules voisines. On garde donc
// un prefixe canonique, repete mot pour mot, et chaque cellule n'ajoute que
// ses helpers propres.
use std::collections::BTreeMap;

#[derive(Clone, Copy, PartialEq, Eq, Debug)]
struct Fract { num: i128, den: i128 }

fn gcd(mut a: i128, mut b: i128) -> i128 {
    while b != 0 { let t = b; b = a % b; a = t; }
    a
}

impl Fract {
    fn new(num: i128, den: i128) -> Self {
        assert!(den != 0, "denominateur nul");
        let mut g = gcd(num.abs(), den.abs());
        if g == 0 { g = 1; }
        let mut n = num / g;
        let mut d = den / g;
        if d < 0 { n = -n; d = -d; }
        Fract { num: n, den: d }
    }
    fn zero() -> Self { Fract { num: 0, den: 1 } }
    fn add(&self, o: &Fract) -> Fract { Fract::new(self.num * o.den + o.num * self.den, self.den * o.den) }
    fn sub(&self, o: &Fract) -> Fract { Fract::new(self.num * o.den - o.num * self.den, self.den * o.den) }
    fn mul(&self, o: &Fract) -> Fract { Fract::new(self.num * o.num, self.den * o.den) }
    fn div(&self, o: &Fract) -> Fract { Fract::new(self.num * o.den, self.den * o.num) }
    fn neg(&self) -> Fract { Fract { num: -self.num, den: self.den } }
    fn is_zero(&self) -> bool { self.num == 0 }
}

impl std::fmt::Display for Fract {
    fn fmt(&self, f: &mut std::fmt::Formatter<'_>) -> std::fmt::Result {
        if self.den == 1 { write!(f, "{}", self.num) } else { write!(f, "{}/{}", self.num, self.den) }
    }
}

#[derive(Clone, Copy, PartialEq, Eq, Debug)]
struct C(Fract, Fract);

fn cmul(x: C, y: C) -> C { C(x.0.mul(&y.0).sub(&x.1.mul(&y.1)), x.0.mul(&y.1).add(&x.1.mul(&y.0))) }
fn cadd(x: C, y: C) -> C { C(x.0.add(&y.0), x.1.add(&y.1)) }
fn cneg(x: C) -> C { C(x.0.neg(), x.1.neg()) }
fn cscale(x: C, n: &Fract) -> C { C(x.0.mul(n), x.1.mul(n)) }

type MatC = Vec<Vec<C>>;
type Mat3 = [[i64; 3]; 3];
type MatN = Vec<Vec<i64>>;

fn mmul(a: &MatC, b: &MatC) -> MatC {
    let mut out = Vec::new();
    for i in 0..a.len() {
        let mut row = Vec::new();
        for j in 0..b[0].len() {
            let mut acc = C(Fract::zero(), Fract::zero());
            for k in 0..b.len() { acc = cadd(acc, cmul(a[i][k], b[k][j])); }
            row.push(acc);
        }
        out.push(row);
    }
    out
}

fn msub(a: &MatC, b: &MatC) -> MatC {
    let mut out = Vec::new();
    for i in 0..a.len() {
        let mut row = Vec::new();
        for j in 0..a[0].len() { row.push(cadd(a[i][j], cneg(b[i][j]))); }
        out.push(row);
    }
    out
}

fn crochet(a: &MatC, b: &MatC) -> MatC { msub(&mmul(a, b), &mmul(b, a)) }

fn meq(a: &MatC, b: &MatC) -> bool {
    (0..a.len()).all(|i| (0..a[0].len()).all(|j| a[i][j] == b[i][j]))
}

fn neg_mat(m: &MatC) -> MatC {
    m.iter().map(|row| row.iter().map(|x| cneg(*x)).collect()).collect()
}

fn mscale2(m: &MatC, k: i64) -> MatC {
    m.iter().map(|row| row.iter().map(|x| cscale(*x, &Fract::new(k as i128, 1))).collect()).collect()
}

fn det3(m: &Mat3) -> i64 {
    m[0][0] * (m[1][1] * m[2][2] - m[1][2] * m[2][1])
        - m[0][1] * (m[1][0] * m[2][2] - m[1][2] * m[2][0])
        + m[0][2] * (m[1][0] * m[2][1] - m[1][1] * m[2][0])
}

fn mul3(a: &Mat3, b: &Mat3) -> Mat3 {
    let mut out = [[0i64; 3]; 3];
    for i in 0..3 {
        for j in 0..3 {
            let mut acc = 0i64;
            for k in 0..3 { acc += a[i][k] * b[k][j]; }
            out[i][j] = acc;
        }
    }
    out
}

const I3: Mat3 = [[1, 0, 0], [0, 1, 0], [0, 0, 1]];

fn ordre3(m: &Mat3) -> i64 {
    let mut r = *m;
    let mut k = 1i64;
    while r != I3 { r = mul3(&r, m); k += 1; }
    k
}

fn py_bool(b: bool) -> String { if b { "True".to_string() } else { "False".to_string() } }

fn py_bools(items: &[bool]) -> String {
    format!("[{}]", items.iter().map(|b| py_bool(*b)).collect::<Vec<String>>().join(", "))
}

fn proportionnel(x: &MatC, base: &[MatC], c0: &C) -> Option<(usize, C)> {
    for (k, g) in base.iter().enumerate() {
        let mut c: Option<C> = None;
        let mut ok = true;
        for i in 0..2usize {
            for j in 0..2usize {
                let a = x[i][j];
                let b = g[i][j];
                if b == *c0 {
                    if a != *c0 { ok = false; }
                } else {
                    let den = b.0.mul(&b.0).add(&b.1.mul(&b.1));
                    let p = a.0.mul(&b.0).add(&a.1.mul(&b.1)).div(&den);
                    let q = a.1.mul(&b.0).sub(&a.0.mul(&b.1)).div(&den);
                    match c {
                        None => c = Some(C(p, q)),
                        Some(cc) => if cmul(cc, b) != a { ok = false; },
                    }
                }
            }
        }
        if ok { if let Some(cc) = c { return Some((k, cc)); } }
    }
    None
}

fn coeff_ad(t: &Vec<MatC>, k: usize, j: usize) -> Vec<i64> {
    let c = crochet(&t[k], &t[j]);
    let mut col = vec![0i64, 0, 0];
    for i in 0..3usize {
        if meq(&c, &t[i]) { col[i] = 1; }
        else if meq(&c, &neg_mat(&t[i])) { col[i] = -1; }
    }
    col
}

fn chi_a1(_g: &Mat3) -> i64 { 1 }
fn chi_a2(g: &Mat3) -> i64 { g[2][2] }
fn chi_e(g: &Mat3) -> i64 { g[0][0] + g[1][1] }
fn chi_dir(g: &Mat3) -> i64 { g[0][0] + g[1][1] + g[2][2] }
fn chi_pair(g: &Mat3) -> i64 { 1 + chi_dir(g) }

fn scalaire_interne(axe: &Vec<Mat3>, f: fn(&Mat3) -> i64, h: fn(&Mat3) -> i64) -> Fract {
    let somme: i128 = axe.iter().map(|g| (f(g) * h(g)) as i128).sum();
    Fract::new(somme, axe.len() as i128)
}

fn commutant(mats: &Vec<MatN>, n: usize) -> usize {
    let mut lignes: Vec<Vec<Fract>> = Vec::new();
    for g in mats.iter() {
        for i in 0..n {
            for j in 0..n {
                let mut ligne = vec![Fract::zero(); n * n];
                for k in 0..n {
                    ligne[n * i + k] = ligne[n * i + k].add(&Fract::new(g[k][j] as i128, 1));
                    ligne[n * k + j] = ligne[n * k + j].sub(&Fract::new(g[i][k] as i128, 1));
                }
                lignes.push(ligne);
            }
        }
    }
    let mut piv = 0usize;
    for col in 0..(n * n) {
        let sel = (piv..lignes.len()).find(|&r| !lignes[r][col].is_zero());
        let sel = match sel { Some(s) => s, None => continue };
        lignes.swap(piv, sel);
        let pv = lignes[piv][col];
        for v in lignes[piv].iter_mut() { *v = v.div(&pv); }
        for r in 0..lignes.len() {
            if r != piv && !lignes[r][col].is_zero() {
                let f = lignes[r][col];
                for c in 0..(n * n) {
                    lignes[r][c] = lignes[r][c].sub(&f.mul(&lignes[piv][c]));
                }
            }
        }
        piv += 1;
    }
    n * n - piv
}

// Demi-tour : exp(pi*T3) = cos(pi/2) I + 2 sin(pi/2) T3 = 2*T3 = -i*sigma_z.


    let C0 = C(Fract::zero(), Fract::zero());
    let C1 = C(Fract::new(1, 1), Fract::zero());
    let CI = C(Fract::zero(), Fract::new(1, 1));

    // Le triple litteral et sa re-orientation (le signe qui retourne est le second).
    let demi = Fract::new(1, 2);
    let g1 = vec![vec![C0, cscale(cneg(CI), &demi)], vec![cscale(cneg(CI), &demi), C0]];
    let g2 = vec![vec![C0, C(demi, Fract::zero())], vec![cneg(C(demi, Fract::zero())), C0]];
    let g3 = vec![vec![cscale(cneg(CI), &demi), C0], vec![C0, cscale(CI, &demi)]];
    let litteral: Vec<MatC> = vec![g1, g2, g3];
    let t: Vec<MatC> = vec![litteral[0].clone(), neg_mat(&litteral[1]), litteral[2].clone()];

    let demi_tour: MatC = t[2].iter()
        .map(|row| row.iter().map(|x| cscale(*x, &Fract::new(2, 1))).collect())
        .collect();
    let mut attendu_demi: MatC = vec![vec![C(Fract::zero(), Fract::new(-1, 1)), C0],
                                      vec![C0, C(Fract::zero(), Fract::new(-1, 1))]];
    attendu_demi[1][1] = C(Fract::zero(), Fract::new(1, 1));
    let demi_ok = meq(&demi_tour, &attendu_demi);
    println!("exp(pi*T3) = 2*T3 = diag(-i, i) = -i*sigma_z -> {}", py_bool(demi_ok));

    // Tour complet : forme fermee cos(pi) = -1, sin(pi) = 0 -> -I. Et comme
    // exp(2*pi*Tj) = (exp(pi*Tj))^2 = (2*Tj)^2 = 4*Tj^2 avec Tj^2 = -(1/4)I mesure
    // ci-dessus, le CARRE du demi-tour est la mesure exacte du tour complet.
    let moins_i: MatC = vec![vec![cneg(C1), C0], vec![C0, cneg(C1)]];
    let exp2pi: Vec<bool> = t.iter().map(|x| meq(&mmul(&mscale2(x, 2), &mscale2(x, 2)), &moins_i)).collect();
    println!("exp(2*pi*Tj) = (2*Tj)^2 = 4*Tj^2 = -I au bout qubit, chaque axe -> {} -> {}",
             py_bools(&exp2pi), py_bool(exp2pi.iter().all(|b| *b)));

    // L'action adjointe sur les directions : ad_T3 en base re-orientee. La mesure
    // porte sur la GENERATRICE T3 (les crochets [T3, T1] = T2, [T3, T2] = -T1,
    // [T3, T3] = 0). L'exponentielle Ad(exp(pi/2 T3)) = exp(pi/2 * ad_T3) est
    // ensuite CALCULEE a partir de cette matrice, pas ecrite a la main.
    let cols: Vec<Vec<i64>> = (0..3).map(|j| coeff_ad(&t, 2, j)).collect();
    let mut ad_t3: Mat3 = [[0i64; 3]; 3];
    for i in 0..3 {
        for j in 0..3 { ad_t3[i][j] = cols[j][i]; }
    }
    println!("ad_T3 (base re-orientée) = {:?}", ad_t3);
    let j_attendu: Mat3 = [[0, -1, 0], [1, 0, 0], [0, 0, 0]];
    println!("ad de la génératrice T3 = J : T1 -> T2, T2 -> -T1, partie planaire du quart de tour -> {}",
             py_bool(ad_t3 == j_attendu));
    // ad_T3 a la structure J : J^3 = -J au bloc planaire, nullite sur l'axe. Donc
    // exp(theta J) = I + sin(theta) J + (1 - cos(theta)) J^2 ; au quart de tour
    // sin = 1 et cos = 0, d'ou exp = I + J + J^2 -- exact, sans flottants.
    let j2 = mul3(&ad_t3, &ad_t3);
    let j3 = mul3(&j2, &ad_t3);
    let j2_attendu: Mat3 = [[-1, 0, 0], [0, -1, 0], [0, 0, 0]];
    let mut j3_attendu: Mat3 = [[0i64; 3]; 3];
    for i in 0..3 {
        for j in 0..3 { j3_attendu[i][j] = -ad_t3[i][j]; }
    }
    println!("ad_T3^2 = diag(-1, -1, 0) -> {} ; ad_T3^3 = -ad_T3 -> {}",
             py_bool(j2 == j2_attendu), py_bool(j3 == j3_attendu));
    let mut quart: Mat3 = [[0i64; 3]; 3];
    for i in 0..3 {
        for j in 0..3 { quart[i][j] = I3[i][j] + ad_t3[i][j] + j2[i][j]; }
    }
    println!("exp(pi/2 * ad_T3) = I + J + J^2 = {:?}", quart);
    let quart_attendu: Mat3 = [[0, -1, 0], [1, 0, 0], [0, 0, 1]];
    println!("Le quart de tour calcule est le plan de rotation applique -> {}",
             py_bool(quart == quart_attendu));
    let q2 = mul3(&quart, &quart);
    let q4 = mul3(&q2, &q2);
    let ad_tour_complet = q4 == I3;
    println!("Le quart de tour sur les directions, à la puissance 4 : identité (quart^4 = I3) -> {}",
             py_bool(ad_tour_complet));
    let double_couvre = demi_ok && exp2pi.iter().all(|b| *b) && ad_tour_complet;
    println!("Double recouvrement : -I au bout qubit, +I sur les directions -> {}",
             py_bool(double_couvre));


exp(pi*T3) = 2*T3 = diag(-i, i) = -i*sigma_z -> True


exp(2*pi*Tj) = (2*Tj)^2 = 4*Tj^2 = -I au bout qubit, chaque axe -> [True, True, True] -> True


ad_T3 (base re-orientée) = [[0, -1, 0], [1, 0, 0], [0, 0, 0]]


ad de la génératrice T3 = J : T1 -> T2, T2 -> -T1, partie planaire du quart de tour -> True


ad_T3^2 = diag(-1, -1, 0) -> True ; ad_T3^3 = -ad_T3 -> True


exp(pi/2 * ad_T3) = I + J + J^2 = [[0, -1, 0], [1, 0, 0], [0, 0, 1]]


Le quart de tour calcule est le plan de rotation applique -> True


Le quart de tour sur les directions, à la puissance 4 : identité (quart^4 = I3) -> True


Double recouvrement : -I au bout qubit, +I sur les directions -> True


### Lecture du résultat (algèbre)

**Mesuré** : les constantes de structure du triple imprimé — $[G_1,G_2] = -G_3$, $[G_2,G_3] = -G_1$, $[G_3,G_1] = -G_2$, toutes les trois au signe moins — et la ré-orientation d'un seul signe qui les amène au symbole alterné $\epsilon_{abc}$ exact ; $T_j^2 = -\tfrac14 I$ ; le demi-tour $-i\sigma_z$ ; le tour complet $-I$ au bout qubit contre $+I$ sur les directions. **Cité** : l'identification de cette algèbre à celle du groupe des rotations de la construction (le stabilisateur de l'unité hermitienne dans les isométries de la scission de la trace), le relevé par conjugaison, et les treize corps typés à chaque bout qubit.

## Les étiquettes de caractères et le commutant

« the directions carry $A_2 \oplus E$ and the pair space of the end $A_1 \oplus A_2 \oplus E$, so its commutant has dimension three. » Les étiquettes sont celles de la littérature (les cinq irreductibles du diédral). Sur **notre** réalisation — le stabilisateur d'axe agissant sur $\mathbb{R}^3$ — nous mesurons les caractères, la décomposition et les commutants : le caractère des directions est exactement le signe de l'axe plus la trace du bloc naturel, et le commutant de l'espace de paire (trivial ⊕ directions) vaut trois, la valeur imprimée.

In [7]:
// Bloc d'outillage commun : identique dans chaque cellule de code du carnet.
// evcxr conserve les items definis d'une cellule a l'autre ; un item de MEME nom
// redefini avec un contenu different casse les cellules voisines. On garde donc
// un prefixe canonique, repete mot pour mot, et chaque cellule n'ajoute que
// ses helpers propres.
use std::collections::BTreeMap;

#[derive(Clone, Copy, PartialEq, Eq, Debug)]
struct Fract { num: i128, den: i128 }

fn gcd(mut a: i128, mut b: i128) -> i128 {
    while b != 0 { let t = b; b = a % b; a = t; }
    a
}

impl Fract {
    fn new(num: i128, den: i128) -> Self {
        assert!(den != 0, "denominateur nul");
        let mut g = gcd(num.abs(), den.abs());
        if g == 0 { g = 1; }
        let mut n = num / g;
        let mut d = den / g;
        if d < 0 { n = -n; d = -d; }
        Fract { num: n, den: d }
    }
    fn zero() -> Self { Fract { num: 0, den: 1 } }
    fn add(&self, o: &Fract) -> Fract { Fract::new(self.num * o.den + o.num * self.den, self.den * o.den) }
    fn sub(&self, o: &Fract) -> Fract { Fract::new(self.num * o.den - o.num * self.den, self.den * o.den) }
    fn mul(&self, o: &Fract) -> Fract { Fract::new(self.num * o.num, self.den * o.den) }
    fn div(&self, o: &Fract) -> Fract { Fract::new(self.num * o.den, self.den * o.num) }
    fn neg(&self) -> Fract { Fract { num: -self.num, den: self.den } }
    fn is_zero(&self) -> bool { self.num == 0 }
}

impl std::fmt::Display for Fract {
    fn fmt(&self, f: &mut std::fmt::Formatter<'_>) -> std::fmt::Result {
        if self.den == 1 { write!(f, "{}", self.num) } else { write!(f, "{}/{}", self.num, self.den) }
    }
}

#[derive(Clone, Copy, PartialEq, Eq, Debug)]
struct C(Fract, Fract);

fn cmul(x: C, y: C) -> C { C(x.0.mul(&y.0).sub(&x.1.mul(&y.1)), x.0.mul(&y.1).add(&x.1.mul(&y.0))) }
fn cadd(x: C, y: C) -> C { C(x.0.add(&y.0), x.1.add(&y.1)) }
fn cneg(x: C) -> C { C(x.0.neg(), x.1.neg()) }
fn cscale(x: C, n: &Fract) -> C { C(x.0.mul(n), x.1.mul(n)) }

type MatC = Vec<Vec<C>>;
type Mat3 = [[i64; 3]; 3];
type MatN = Vec<Vec<i64>>;

fn mmul(a: &MatC, b: &MatC) -> MatC {
    let mut out = Vec::new();
    for i in 0..a.len() {
        let mut row = Vec::new();
        for j in 0..b[0].len() {
            let mut acc = C(Fract::zero(), Fract::zero());
            for k in 0..b.len() { acc = cadd(acc, cmul(a[i][k], b[k][j])); }
            row.push(acc);
        }
        out.push(row);
    }
    out
}

fn msub(a: &MatC, b: &MatC) -> MatC {
    let mut out = Vec::new();
    for i in 0..a.len() {
        let mut row = Vec::new();
        for j in 0..a[0].len() { row.push(cadd(a[i][j], cneg(b[i][j]))); }
        out.push(row);
    }
    out
}

fn crochet(a: &MatC, b: &MatC) -> MatC { msub(&mmul(a, b), &mmul(b, a)) }

fn meq(a: &MatC, b: &MatC) -> bool {
    (0..a.len()).all(|i| (0..a[0].len()).all(|j| a[i][j] == b[i][j]))
}

fn neg_mat(m: &MatC) -> MatC {
    m.iter().map(|row| row.iter().map(|x| cneg(*x)).collect()).collect()
}

fn mscale2(m: &MatC, k: i64) -> MatC {
    m.iter().map(|row| row.iter().map(|x| cscale(*x, &Fract::new(k as i128, 1))).collect()).collect()
}

fn det3(m: &Mat3) -> i64 {
    m[0][0] * (m[1][1] * m[2][2] - m[1][2] * m[2][1])
        - m[0][1] * (m[1][0] * m[2][2] - m[1][2] * m[2][0])
        + m[0][2] * (m[1][0] * m[2][1] - m[1][1] * m[2][0])
}

fn mul3(a: &Mat3, b: &Mat3) -> Mat3 {
    let mut out = [[0i64; 3]; 3];
    for i in 0..3 {
        for j in 0..3 {
            let mut acc = 0i64;
            for k in 0..3 { acc += a[i][k] * b[k][j]; }
            out[i][j] = acc;
        }
    }
    out
}

const I3: Mat3 = [[1, 0, 0], [0, 1, 0], [0, 0, 1]];

fn ordre3(m: &Mat3) -> i64 {
    let mut r = *m;
    let mut k = 1i64;
    while r != I3 { r = mul3(&r, m); k += 1; }
    k
}

fn py_bool(b: bool) -> String { if b { "True".to_string() } else { "False".to_string() } }

fn py_bools(items: &[bool]) -> String {
    format!("[{}]", items.iter().map(|b| py_bool(*b)).collect::<Vec<String>>().join(", "))
}

fn proportionnel(x: &MatC, base: &[MatC], c0: &C) -> Option<(usize, C)> {
    for (k, g) in base.iter().enumerate() {
        let mut c: Option<C> = None;
        let mut ok = true;
        for i in 0..2usize {
            for j in 0..2usize {
                let a = x[i][j];
                let b = g[i][j];
                if b == *c0 {
                    if a != *c0 { ok = false; }
                } else {
                    let den = b.0.mul(&b.0).add(&b.1.mul(&b.1));
                    let p = a.0.mul(&b.0).add(&a.1.mul(&b.1)).div(&den);
                    let q = a.1.mul(&b.0).sub(&a.0.mul(&b.1)).div(&den);
                    match c {
                        None => c = Some(C(p, q)),
                        Some(cc) => if cmul(cc, b) != a { ok = false; },
                    }
                }
            }
        }
        if ok { if let Some(cc) = c { return Some((k, cc)); } }
    }
    None
}

fn coeff_ad(t: &Vec<MatC>, k: usize, j: usize) -> Vec<i64> {
    let c = crochet(&t[k], &t[j]);
    let mut col = vec![0i64, 0, 0];
    for i in 0..3usize {
        if meq(&c, &t[i]) { col[i] = 1; }
        else if meq(&c, &neg_mat(&t[i])) { col[i] = -1; }
    }
    col
}

fn chi_a1(_g: &Mat3) -> i64 { 1 }
fn chi_a2(g: &Mat3) -> i64 { g[2][2] }
fn chi_e(g: &Mat3) -> i64 { g[0][0] + g[1][1] }
fn chi_dir(g: &Mat3) -> i64 { g[0][0] + g[1][1] + g[2][2] }
fn chi_pair(g: &Mat3) -> i64 { 1 + chi_dir(g) }

fn scalaire_interne(axe: &Vec<Mat3>, f: fn(&Mat3) -> i64, h: fn(&Mat3) -> i64) -> Fract {
    let somme: i128 = axe.iter().map(|g| (f(g) * h(g)) as i128).sum();
    Fract::new(somme, axe.len() as i128)
}

fn commutant(mats: &Vec<MatN>, n: usize) -> usize {
    let mut lignes: Vec<Vec<Fract>> = Vec::new();
    for g in mats.iter() {
        for i in 0..n {
            for j in 0..n {
                let mut ligne = vec![Fract::zero(); n * n];
                for k in 0..n {
                    ligne[n * i + k] = ligne[n * i + k].add(&Fract::new(g[k][j] as i128, 1));
                    ligne[n * k + j] = ligne[n * k + j].sub(&Fract::new(g[i][k] as i128, 1));
                }
                lignes.push(ligne);
            }
        }
    }
    let mut piv = 0usize;
    for col in 0..(n * n) {
        let sel = (piv..lignes.len()).find(|&r| !lignes[r][col].is_zero());
        let sel = match sel { Some(s) => s, None => continue };
        lignes.swap(piv, sel);
        let pv = lignes[piv][col];
        for v in lignes[piv].iter_mut() { *v = v.div(&pv); }
        for r in 0..lignes.len() {
            if r != piv && !lignes[r][col].is_zero() {
                let f = lignes[r][col];
                for c in 0..(n * n) {
                    lignes[r][c] = lignes[r][c].sub(&f.mul(&lignes[piv][c]));
                }
            }
        }
        piv += 1;
    }
    n * n - piv
}

// Caracteres sur notre realisation (etiquettes citees, decomposition mesuree).


    let perms: [[usize; 3]; 6] = [
        [0, 1, 2], [0, 2, 1], [1, 0, 2], [1, 2, 0], [2, 0, 1], [2, 1, 0],
    ];
    let signes: [[i64; 3]; 8] = [
        [1, 1, 1], [1, 1, -1], [1, -1, 1], [1, -1, -1],
        [-1, 1, 1], [-1, 1, -1], [-1, -1, 1], [-1, -1, -1],
    ];
    let mut rotations24: Vec<Mat3> = Vec::new();
    for perm in perms.iter() {
        for sg in signes.iter() {
            let mut m = [[0i64; 3]; 3];
            for i in 0..3 { m[i][perm[i]] = sg[i]; }
            if det3(&m) == 1 { rotations24.push(m); }
        }
    }
    let axe: Vec<Mat3> = rotations24.iter()
        .filter(|m| m[0][2] == 0 && m[1][2] == 0 && m[2][0] == 0 && m[2][1] == 0)
        .cloned()
        .collect();


    println!("<chi_dir, A1> = {} (aucune composante triviale)", scalaire_interne(&axe, chi_dir, chi_a1));
    println!("<chi_dir, A2> = {} ; <chi_dir, E> = {}", scalaire_interne(&axe, chi_dir, chi_a2), scalaire_interne(&axe, chi_dir, chi_e));
    println!("<chi_pair, A1> = {} ; <chi_pair, A2> = {} ; <chi_pair, E> = {}",
             scalaire_interne(&axe, chi_pair, chi_a1), scalaire_interne(&axe, chi_pair, chi_a2), scalaire_interne(&axe, chi_pair, chi_e));

    // Commutants : dimension de l'espace des matrices qui commutent a tout le groupe.
    // L'elimination est MENEE EN RATIONNELS EXACTS (le source la menait en flottants ;
    // le rang, lui, ne depend pas du pivot).

    let mut axe_n: Vec<MatN> = Vec::new();
    for g in axe.iter() {
        axe_n.push((0..3).map(|i| (0..3).map(|j| g[i][j]).collect()).collect());
    }
    let mut pair4: Vec<MatN> = Vec::new();
    for g in axe.iter() {
        let mut b = vec![vec![0i64; 4]; 4];
        b[0][0] = 1;
        for i in 0..3 {
            for j in 0..3 { b[1 + i][1 + j] = g[i][j]; }
        }
        pair4.push(b);
    }
    let d_dir = commutant(&axe_n, 3);
    let d_pair = commutant(&pair4, 4);
    println!("Commutant du modele directions (A2+E) : {} ; du modele espace de paire (A1+A2+E) : {} -> la valeur trois imprimee est celle de l'espace de paire sur ce modele",
             d_dir, d_pair);


<chi_dir, A1> = 0 (aucune composante triviale)


<chi_dir, A2> = 1 ; <chi_dir, E> = 1


<chi_pair, A1> = 1 ; <chi_pair, A2> = 1 ; <chi_pair, E> = 1


Commutant du modele directions (A2+E) : 2 ; du modele espace de paire (A1+A2+E) : 3 -> la valeur trois imprimee est celle de l'espace de paire sur ce modele


### Lecture du résultat (caractères)

**Mesuré** : sur notre réalisation, le caractère des directions se décompose en $A_2 \oplus E$ (produits scalaires 0, 1, 1) et celui de l'espace de paire en $A_1 \oplus A_2 \oplus E$ (1, 1, 1) ; le commutant de l'espace de paire vaut **trois**, celui des directions deux. La phrase imprimée — « so its commutant has dimension three » — suit immédiatement les deux étiquettes ; la valeur trois est, sur notre modèle, celle de **l'espace de paire** ($1 + 1 + 1$), ce qui est aussi la lecture grammaticale. **Cité** : les étiquettes elles-mêmes, attachées à la construction de la source.

## Ce que le census porte (cité)

Le census du chapitre enumere les 102 bouts qubit des corps de la construction — sites, liens et types de paire — et verifie le stabilisateur a chacun. Nous citons sa table et ses comptes : ils ne se rejouent pas ici (la liste des 102 objets n'est pas imprimee dans l'extrait).

In [8]:
// Bloc d'outillage commun : identique dans chaque cellule de code du carnet.
// evcxr conserve les items definis d'une cellule a l'autre ; un item de MEME nom
// redefini avec un contenu different casse les cellules voisines. On garde donc
// un prefixe canonique, repete mot pour mot, et chaque cellule n'ajoute que
// ses helpers propres.
use std::collections::BTreeMap;

#[derive(Clone, Copy, PartialEq, Eq, Debug)]
struct Fract { num: i128, den: i128 }

fn gcd(mut a: i128, mut b: i128) -> i128 {
    while b != 0 { let t = b; b = a % b; a = t; }
    a
}

impl Fract {
    fn new(num: i128, den: i128) -> Self {
        assert!(den != 0, "denominateur nul");
        let mut g = gcd(num.abs(), den.abs());
        if g == 0 { g = 1; }
        let mut n = num / g;
        let mut d = den / g;
        if d < 0 { n = -n; d = -d; }
        Fract { num: n, den: d }
    }
    fn zero() -> Self { Fract { num: 0, den: 1 } }
    fn add(&self, o: &Fract) -> Fract { Fract::new(self.num * o.den + o.num * self.den, self.den * o.den) }
    fn sub(&self, o: &Fract) -> Fract { Fract::new(self.num * o.den - o.num * self.den, self.den * o.den) }
    fn mul(&self, o: &Fract) -> Fract { Fract::new(self.num * o.num, self.den * o.den) }
    fn div(&self, o: &Fract) -> Fract { Fract::new(self.num * o.den, self.den * o.num) }
    fn neg(&self) -> Fract { Fract { num: -self.num, den: self.den } }
    fn is_zero(&self) -> bool { self.num == 0 }
}

impl std::fmt::Display for Fract {
    fn fmt(&self, f: &mut std::fmt::Formatter<'_>) -> std::fmt::Result {
        if self.den == 1 { write!(f, "{}", self.num) } else { write!(f, "{}/{}", self.num, self.den) }
    }
}

#[derive(Clone, Copy, PartialEq, Eq, Debug)]
struct C(Fract, Fract);

fn cmul(x: C, y: C) -> C { C(x.0.mul(&y.0).sub(&x.1.mul(&y.1)), x.0.mul(&y.1).add(&x.1.mul(&y.0))) }
fn cadd(x: C, y: C) -> C { C(x.0.add(&y.0), x.1.add(&y.1)) }
fn cneg(x: C) -> C { C(x.0.neg(), x.1.neg()) }
fn cscale(x: C, n: &Fract) -> C { C(x.0.mul(n), x.1.mul(n)) }

type MatC = Vec<Vec<C>>;
type Mat3 = [[i64; 3]; 3];
type MatN = Vec<Vec<i64>>;

fn mmul(a: &MatC, b: &MatC) -> MatC {
    let mut out = Vec::new();
    for i in 0..a.len() {
        let mut row = Vec::new();
        for j in 0..b[0].len() {
            let mut acc = C(Fract::zero(), Fract::zero());
            for k in 0..b.len() { acc = cadd(acc, cmul(a[i][k], b[k][j])); }
            row.push(acc);
        }
        out.push(row);
    }
    out
}

fn msub(a: &MatC, b: &MatC) -> MatC {
    let mut out = Vec::new();
    for i in 0..a.len() {
        let mut row = Vec::new();
        for j in 0..a[0].len() { row.push(cadd(a[i][j], cneg(b[i][j]))); }
        out.push(row);
    }
    out
}

fn crochet(a: &MatC, b: &MatC) -> MatC { msub(&mmul(a, b), &mmul(b, a)) }

fn meq(a: &MatC, b: &MatC) -> bool {
    (0..a.len()).all(|i| (0..a[0].len()).all(|j| a[i][j] == b[i][j]))
}

fn neg_mat(m: &MatC) -> MatC {
    m.iter().map(|row| row.iter().map(|x| cneg(*x)).collect()).collect()
}

fn mscale2(m: &MatC, k: i64) -> MatC {
    m.iter().map(|row| row.iter().map(|x| cscale(*x, &Fract::new(k as i128, 1))).collect()).collect()
}

fn det3(m: &Mat3) -> i64 {
    m[0][0] * (m[1][1] * m[2][2] - m[1][2] * m[2][1])
        - m[0][1] * (m[1][0] * m[2][2] - m[1][2] * m[2][0])
        + m[0][2] * (m[1][0] * m[2][1] - m[1][1] * m[2][0])
}

fn mul3(a: &Mat3, b: &Mat3) -> Mat3 {
    let mut out = [[0i64; 3]; 3];
    for i in 0..3 {
        for j in 0..3 {
            let mut acc = 0i64;
            for k in 0..3 { acc += a[i][k] * b[k][j]; }
            out[i][j] = acc;
        }
    }
    out
}

const I3: Mat3 = [[1, 0, 0], [0, 1, 0], [0, 0, 1]];

fn ordre3(m: &Mat3) -> i64 {
    let mut r = *m;
    let mut k = 1i64;
    while r != I3 { r = mul3(&r, m); k += 1; }
    k
}

fn py_bool(b: bool) -> String { if b { "True".to_string() } else { "False".to_string() } }

fn py_bools(items: &[bool]) -> String {
    format!("[{}]", items.iter().map(|b| py_bool(*b)).collect::<Vec<String>>().join(", "))
}

fn proportionnel(x: &MatC, base: &[MatC], c0: &C) -> Option<(usize, C)> {
    for (k, g) in base.iter().enumerate() {
        let mut c: Option<C> = None;
        let mut ok = true;
        for i in 0..2usize {
            for j in 0..2usize {
                let a = x[i][j];
                let b = g[i][j];
                if b == *c0 {
                    if a != *c0 { ok = false; }
                } else {
                    let den = b.0.mul(&b.0).add(&b.1.mul(&b.1));
                    let p = a.0.mul(&b.0).add(&a.1.mul(&b.1)).div(&den);
                    let q = a.1.mul(&b.0).sub(&a.0.mul(&b.1)).div(&den);
                    match c {
                        None => c = Some(C(p, q)),
                        Some(cc) => if cmul(cc, b) != a { ok = false; },
                    }
                }
            }
        }
        if ok { if let Some(cc) = c { return Some((k, cc)); } }
    }
    None
}

fn coeff_ad(t: &Vec<MatC>, k: usize, j: usize) -> Vec<i64> {
    let c = crochet(&t[k], &t[j]);
    let mut col = vec![0i64, 0, 0];
    for i in 0..3usize {
        if meq(&c, &t[i]) { col[i] = 1; }
        else if meq(&c, &neg_mat(&t[i])) { col[i] = -1; }
    }
    col
}

fn chi_a1(_g: &Mat3) -> i64 { 1 }
fn chi_a2(g: &Mat3) -> i64 { g[2][2] }
fn chi_e(g: &Mat3) -> i64 { g[0][0] + g[1][1] }
fn chi_dir(g: &Mat3) -> i64 { g[0][0] + g[1][1] + g[2][2] }
fn chi_pair(g: &Mat3) -> i64 { 1 + chi_dir(g) }

fn scalaire_interne(axe: &Vec<Mat3>, f: fn(&Mat3) -> i64, h: fn(&Mat3) -> i64) -> Fract {
    let somme: i128 = axe.iter().map(|g| (f(g) * h(g)) as i128).sum();
    Fract::new(somme, axe.len() as i128)
}

fn commutant(mats: &Vec<MatN>, n: usize) -> usize {
    let mut lignes: Vec<Vec<Fract>> = Vec::new();
    for g in mats.iter() {
        for i in 0..n {
            for j in 0..n {
                let mut ligne = vec![Fract::zero(); n * n];
                for k in 0..n {
                    ligne[n * i + k] = ligne[n * i + k].add(&Fract::new(g[k][j] as i128, 1));
                    ligne[n * k + j] = ligne[n * k + j].sub(&Fract::new(g[i][k] as i128, 1));
                }
                lignes.push(ligne);
            }
        }
    }
    let mut piv = 0usize;
    for col in 0..(n * n) {
        let sel = (piv..lignes.len()).find(|&r| !lignes[r][col].is_zero());
        let sel = match sel { Some(s) => s, None => continue };
        lignes.swap(piv, sel);
        let pv = lignes[piv][col];
        for v in lignes[piv].iter_mut() { *v = v.div(&pv); }
        for r in 0..lignes.len() {
            if r != piv && !lignes[r][col].is_zero() {
                let f = lignes[r][col];
                for c in 0..(n * n) {
                    lignes[r][c] = lignes[r][c].sub(&f.mul(&lignes[piv][c]));
                }
            }
        }
        piv += 1;
    }
    n * n - piv
}

// Les comptes du census, cites tel qu'imprimes (extrait p. 143).


    // Donnees citees : l'ordre d'insertion du dictionnaire du source.
    let census: [(&str, i64); 5] = [
        ("bouts qubit enumeres", 102),
        ("corps de la construction types", 13),
        ("controles lus pareil aux deux moteurs", 13),
        ("contre-tests aux lettres epinglees", 5),
        ("longueur de la plus longue chaine de premices", 12),
    ];
    for (k, v) in census.iter() {
        println!("{} : {} (CITÉ)", k, v);
    }
    println!("Contre-test nomme : uniterre frappee -> plus de groupe, une algebre d'isometrie de dimension trois (cite)");
    println!("Second moteur construit a part : chaque cellule reproduite egale (cite)");


bouts qubit enumeres : 102 (CITÉ)


corps de la construction types : 13 (CITÉ)


controles lus pareil aux deux moteurs : 13 (CITÉ)


contre-tests aux lettres epinglees : 5 (CITÉ)


longueur de la plus longue chaine de premices : 12 (CITÉ)


Contre-test nomme : uniterre frappee -> plus de groupe, une algebre d'isometrie de dimension trois (cite)


Second moteur construit a part : chaque cellule reproduite egale (cite)


## Carrefours avec la série

- **Chaînes-09** : l'éq. 76 mesurait $\tau = 1/4$, réciproque du rapport des carrés — les **quatre** cartes de signe au rapport carré. Ici les phases du groupe d'identification sont les **racines quatrièmes** de l'unité, et les quatre signes de l'éq. 76 sont le sous-groupe d'ordre quatre des phases réelles. La structure « quart » est la même des deux côtés, à des objets différents.
- **Chaînes-01** : le lien qutrit-qubit dont l'algèbre de cadre porte les directions dérivées — le même objet dont l'ordre coordinate lisait le centre unidimensionnel.
- **Chaînes-05** : les niveaux du ch. 44 étaient lus au corps électron du lien $(2,3)$ frappé ; les docks typés ici (propre au bout proche, impropre au bout loin) sont ceux de la même construction des corps.

## Ouvert — ligne à ligne

1. **Une direction par paire depuis le support partagé** : une affirmation d'unicité « where the construction admits several readings, decided at one of four and superseded by the automorphism theorem » (§53) — l'un des quatre choix est fait, la lecture automorphique le remplace, l'unicité reste un résultat ouvert.
2. **Le signe de site propre** : « equals the standing swap sign at seventeen of the eighteen objects », parting au tau — son site propre est un port qubit-qubit apparié à une feuille que le transport n'ouvre à aucune orientation. Le parting est nommé, non expliqué : ouvert.
3. **L'étiquette orbitale** conservée à chaque instance d'interaction — **conditionnelle** sur la clause à un acte (une prémisse de la série, pas un théorème établi).
4. **La rotation d'angle générique** est hors du groupe — le continu est fermé au premier résultat du chapitre : la construction ne livre que le groupe fini et son algèbre, pas SO(3) tout entier ; c'est une limite de l'objet, pas une lacune de preuve.
5. **Le census des 102 bouts** : cité, non rejoué — la table d'objets n'est pas imprimée dans l'extrait ; nos comptes mesurés portent les groupes finis et l'algèbre, pas l'attribution typée corps par corps.

### Exercice 1 — l'autre moitié du réseau

Les 48 permutations signées se partagent en 24 de déterminant +1 (les rotations, comptées ci-dessus) et 24 de déterminant −1. Énumérer les secondes et vérifier le partage.

In [9]:
// Bloc d'outillage commun : identique dans chaque cellule de code du carnet.
// evcxr conserve les items definis d'une cellule a l'autre ; un item de MEME nom
// redefini avec un contenu different casse les cellules voisines. On garde donc
// un prefixe canonique, repete mot pour mot, et chaque cellule n'ajoute que
// ses helpers propres.
use std::collections::BTreeMap;

#[derive(Clone, Copy, PartialEq, Eq, Debug)]
struct Fract { num: i128, den: i128 }

fn gcd(mut a: i128, mut b: i128) -> i128 {
    while b != 0 { let t = b; b = a % b; a = t; }
    a
}

impl Fract {
    fn new(num: i128, den: i128) -> Self {
        assert!(den != 0, "denominateur nul");
        let mut g = gcd(num.abs(), den.abs());
        if g == 0 { g = 1; }
        let mut n = num / g;
        let mut d = den / g;
        if d < 0 { n = -n; d = -d; }
        Fract { num: n, den: d }
    }
    fn zero() -> Self { Fract { num: 0, den: 1 } }
    fn add(&self, o: &Fract) -> Fract { Fract::new(self.num * o.den + o.num * self.den, self.den * o.den) }
    fn sub(&self, o: &Fract) -> Fract { Fract::new(self.num * o.den - o.num * self.den, self.den * o.den) }
    fn mul(&self, o: &Fract) -> Fract { Fract::new(self.num * o.num, self.den * o.den) }
    fn div(&self, o: &Fract) -> Fract { Fract::new(self.num * o.den, self.den * o.num) }
    fn neg(&self) -> Fract { Fract { num: -self.num, den: self.den } }
    fn is_zero(&self) -> bool { self.num == 0 }
}

impl std::fmt::Display for Fract {
    fn fmt(&self, f: &mut std::fmt::Formatter<'_>) -> std::fmt::Result {
        if self.den == 1 { write!(f, "{}", self.num) } else { write!(f, "{}/{}", self.num, self.den) }
    }
}

#[derive(Clone, Copy, PartialEq, Eq, Debug)]
struct C(Fract, Fract);

fn cmul(x: C, y: C) -> C { C(x.0.mul(&y.0).sub(&x.1.mul(&y.1)), x.0.mul(&y.1).add(&x.1.mul(&y.0))) }
fn cadd(x: C, y: C) -> C { C(x.0.add(&y.0), x.1.add(&y.1)) }
fn cneg(x: C) -> C { C(x.0.neg(), x.1.neg()) }
fn cscale(x: C, n: &Fract) -> C { C(x.0.mul(n), x.1.mul(n)) }

type MatC = Vec<Vec<C>>;
type Mat3 = [[i64; 3]; 3];
type MatN = Vec<Vec<i64>>;

fn mmul(a: &MatC, b: &MatC) -> MatC {
    let mut out = Vec::new();
    for i in 0..a.len() {
        let mut row = Vec::new();
        for j in 0..b[0].len() {
            let mut acc = C(Fract::zero(), Fract::zero());
            for k in 0..b.len() { acc = cadd(acc, cmul(a[i][k], b[k][j])); }
            row.push(acc);
        }
        out.push(row);
    }
    out
}

fn msub(a: &MatC, b: &MatC) -> MatC {
    let mut out = Vec::new();
    for i in 0..a.len() {
        let mut row = Vec::new();
        for j in 0..a[0].len() { row.push(cadd(a[i][j], cneg(b[i][j]))); }
        out.push(row);
    }
    out
}

fn crochet(a: &MatC, b: &MatC) -> MatC { msub(&mmul(a, b), &mmul(b, a)) }

fn meq(a: &MatC, b: &MatC) -> bool {
    (0..a.len()).all(|i| (0..a[0].len()).all(|j| a[i][j] == b[i][j]))
}

fn neg_mat(m: &MatC) -> MatC {
    m.iter().map(|row| row.iter().map(|x| cneg(*x)).collect()).collect()
}

fn mscale2(m: &MatC, k: i64) -> MatC {
    m.iter().map(|row| row.iter().map(|x| cscale(*x, &Fract::new(k as i128, 1))).collect()).collect()
}

fn det3(m: &Mat3) -> i64 {
    m[0][0] * (m[1][1] * m[2][2] - m[1][2] * m[2][1])
        - m[0][1] * (m[1][0] * m[2][2] - m[1][2] * m[2][0])
        + m[0][2] * (m[1][0] * m[2][1] - m[1][1] * m[2][0])
}

fn mul3(a: &Mat3, b: &Mat3) -> Mat3 {
    let mut out = [[0i64; 3]; 3];
    for i in 0..3 {
        for j in 0..3 {
            let mut acc = 0i64;
            for k in 0..3 { acc += a[i][k] * b[k][j]; }
            out[i][j] = acc;
        }
    }
    out
}

const I3: Mat3 = [[1, 0, 0], [0, 1, 0], [0, 0, 1]];

fn ordre3(m: &Mat3) -> i64 {
    let mut r = *m;
    let mut k = 1i64;
    while r != I3 { r = mul3(&r, m); k += 1; }
    k
}

fn py_bool(b: bool) -> String { if b { "True".to_string() } else { "False".to_string() } }

fn py_bools(items: &[bool]) -> String {
    format!("[{}]", items.iter().map(|b| py_bool(*b)).collect::<Vec<String>>().join(", "))
}

fn proportionnel(x: &MatC, base: &[MatC], c0: &C) -> Option<(usize, C)> {
    for (k, g) in base.iter().enumerate() {
        let mut c: Option<C> = None;
        let mut ok = true;
        for i in 0..2usize {
            for j in 0..2usize {
                let a = x[i][j];
                let b = g[i][j];
                if b == *c0 {
                    if a != *c0 { ok = false; }
                } else {
                    let den = b.0.mul(&b.0).add(&b.1.mul(&b.1));
                    let p = a.0.mul(&b.0).add(&a.1.mul(&b.1)).div(&den);
                    let q = a.1.mul(&b.0).sub(&a.0.mul(&b.1)).div(&den);
                    match c {
                        None => c = Some(C(p, q)),
                        Some(cc) => if cmul(cc, b) != a { ok = false; },
                    }
                }
            }
        }
        if ok { if let Some(cc) = c { return Some((k, cc)); } }
    }
    None
}

fn coeff_ad(t: &Vec<MatC>, k: usize, j: usize) -> Vec<i64> {
    let c = crochet(&t[k], &t[j]);
    let mut col = vec![0i64, 0, 0];
    for i in 0..3usize {
        if meq(&c, &t[i]) { col[i] = 1; }
        else if meq(&c, &neg_mat(&t[i])) { col[i] = -1; }
    }
    col
}

fn chi_a1(_g: &Mat3) -> i64 { 1 }
fn chi_a2(g: &Mat3) -> i64 { g[2][2] }
fn chi_e(g: &Mat3) -> i64 { g[0][0] + g[1][1] }
fn chi_dir(g: &Mat3) -> i64 { g[0][0] + g[1][1] + g[2][2] }
fn chi_pair(g: &Mat3) -> i64 { 1 + chi_dir(g) }

fn scalaire_interne(axe: &Vec<Mat3>, f: fn(&Mat3) -> i64, h: fn(&Mat3) -> i64) -> Fract {
    let somme: i128 = axe.iter().map(|g| (f(g) * h(g)) as i128).sum();
    Fract::new(somme, axe.len() as i128)
}

fn commutant(mats: &Vec<MatN>, n: usize) -> usize {
    let mut lignes: Vec<Vec<Fract>> = Vec::new();
    for g in mats.iter() {
        for i in 0..n {
            for j in 0..n {
                let mut ligne = vec![Fract::zero(); n * n];
                for k in 0..n {
                    ligne[n * i + k] = ligne[n * i + k].add(&Fract::new(g[k][j] as i128, 1));
                    ligne[n * k + j] = ligne[n * k + j].sub(&Fract::new(g[i][k] as i128, 1));
                }
                lignes.push(ligne);
            }
        }
    }
    let mut piv = 0usize;
    for col in 0..(n * n) {
        let sel = (piv..lignes.len()).find(|&r| !lignes[r][col].is_zero());
        let sel = match sel { Some(s) => s, None => continue };
        lignes.swap(piv, sel);
        let pv = lignes[piv][col];
        for v in lignes[piv].iter_mut() { *v = v.div(&pv); }
        for r in 0..lignes.len() {
            if r != piv && !lignes[r][col].is_zero() {
                let f = lignes[r][col];
                for c in 0..(n * n) {
                    lignes[r][c] = lignes[r][c].sub(&f.mul(&lignes[piv][c]));
                }
            }
        }
        piv += 1;
    }
    n * n - piv
}

// Etendre l'enumeration aux permutations signees de determinant -1.


    // TODO etudiant : compter les permutations signees 3x3 de det = -1
    // et verifier 24 + 24 = 48 (les « improprement typés » du census sont a lire dans ce miroir)
    let _impropres: Option<Vec<Vec<Vec<i64>>>> = None;
    println!("Exercice a completer");


Exercice a completer


### Exercice 2 — un autre point rationnel

L'exponentielle exacte vit aux points rationnels du cercle. Au point pythagoricien $(\cos\theta, \sin\theta) = (5/13, 12/13)$, écrire la matrice de rotation de l'action adjointe dans le plan $(T_1, T_2)$ et vérifier exactement qu'elle préserve la forme euclidienne.

In [10]:
// Bloc d'outillage commun : identique dans chaque cellule de code du carnet.
// evcxr conserve les items definis d'une cellule a l'autre ; un item de MEME nom
// redefini avec un contenu different casse les cellules voisines. On garde donc
// un prefixe canonique, repete mot pour mot, et chaque cellule n'ajoute que
// ses helpers propres.
use std::collections::BTreeMap;

#[derive(Clone, Copy, PartialEq, Eq, Debug)]
struct Fract { num: i128, den: i128 }

fn gcd(mut a: i128, mut b: i128) -> i128 {
    while b != 0 { let t = b; b = a % b; a = t; }
    a
}

impl Fract {
    fn new(num: i128, den: i128) -> Self {
        assert!(den != 0, "denominateur nul");
        let mut g = gcd(num.abs(), den.abs());
        if g == 0 { g = 1; }
        let mut n = num / g;
        let mut d = den / g;
        if d < 0 { n = -n; d = -d; }
        Fract { num: n, den: d }
    }
    fn zero() -> Self { Fract { num: 0, den: 1 } }
    fn add(&self, o: &Fract) -> Fract { Fract::new(self.num * o.den + o.num * self.den, self.den * o.den) }
    fn sub(&self, o: &Fract) -> Fract { Fract::new(self.num * o.den - o.num * self.den, self.den * o.den) }
    fn mul(&self, o: &Fract) -> Fract { Fract::new(self.num * o.num, self.den * o.den) }
    fn div(&self, o: &Fract) -> Fract { Fract::new(self.num * o.den, self.den * o.num) }
    fn neg(&self) -> Fract { Fract { num: -self.num, den: self.den } }
    fn is_zero(&self) -> bool { self.num == 0 }
}

impl std::fmt::Display for Fract {
    fn fmt(&self, f: &mut std::fmt::Formatter<'_>) -> std::fmt::Result {
        if self.den == 1 { write!(f, "{}", self.num) } else { write!(f, "{}/{}", self.num, self.den) }
    }
}

#[derive(Clone, Copy, PartialEq, Eq, Debug)]
struct C(Fract, Fract);

fn cmul(x: C, y: C) -> C { C(x.0.mul(&y.0).sub(&x.1.mul(&y.1)), x.0.mul(&y.1).add(&x.1.mul(&y.0))) }
fn cadd(x: C, y: C) -> C { C(x.0.add(&y.0), x.1.add(&y.1)) }
fn cneg(x: C) -> C { C(x.0.neg(), x.1.neg()) }
fn cscale(x: C, n: &Fract) -> C { C(x.0.mul(n), x.1.mul(n)) }

type MatC = Vec<Vec<C>>;
type Mat3 = [[i64; 3]; 3];
type MatN = Vec<Vec<i64>>;

fn mmul(a: &MatC, b: &MatC) -> MatC {
    let mut out = Vec::new();
    for i in 0..a.len() {
        let mut row = Vec::new();
        for j in 0..b[0].len() {
            let mut acc = C(Fract::zero(), Fract::zero());
            for k in 0..b.len() { acc = cadd(acc, cmul(a[i][k], b[k][j])); }
            row.push(acc);
        }
        out.push(row);
    }
    out
}

fn msub(a: &MatC, b: &MatC) -> MatC {
    let mut out = Vec::new();
    for i in 0..a.len() {
        let mut row = Vec::new();
        for j in 0..a[0].len() { row.push(cadd(a[i][j], cneg(b[i][j]))); }
        out.push(row);
    }
    out
}

fn crochet(a: &MatC, b: &MatC) -> MatC { msub(&mmul(a, b), &mmul(b, a)) }

fn meq(a: &MatC, b: &MatC) -> bool {
    (0..a.len()).all(|i| (0..a[0].len()).all(|j| a[i][j] == b[i][j]))
}

fn neg_mat(m: &MatC) -> MatC {
    m.iter().map(|row| row.iter().map(|x| cneg(*x)).collect()).collect()
}

fn mscale2(m: &MatC, k: i64) -> MatC {
    m.iter().map(|row| row.iter().map(|x| cscale(*x, &Fract::new(k as i128, 1))).collect()).collect()
}

fn det3(m: &Mat3) -> i64 {
    m[0][0] * (m[1][1] * m[2][2] - m[1][2] * m[2][1])
        - m[0][1] * (m[1][0] * m[2][2] - m[1][2] * m[2][0])
        + m[0][2] * (m[1][0] * m[2][1] - m[1][1] * m[2][0])
}

fn mul3(a: &Mat3, b: &Mat3) -> Mat3 {
    let mut out = [[0i64; 3]; 3];
    for i in 0..3 {
        for j in 0..3 {
            let mut acc = 0i64;
            for k in 0..3 { acc += a[i][k] * b[k][j]; }
            out[i][j] = acc;
        }
    }
    out
}

const I3: Mat3 = [[1, 0, 0], [0, 1, 0], [0, 0, 1]];

fn ordre3(m: &Mat3) -> i64 {
    let mut r = *m;
    let mut k = 1i64;
    while r != I3 { r = mul3(&r, m); k += 1; }
    k
}

fn py_bool(b: bool) -> String { if b { "True".to_string() } else { "False".to_string() } }

fn py_bools(items: &[bool]) -> String {
    format!("[{}]", items.iter().map(|b| py_bool(*b)).collect::<Vec<String>>().join(", "))
}

fn proportionnel(x: &MatC, base: &[MatC], c0: &C) -> Option<(usize, C)> {
    for (k, g) in base.iter().enumerate() {
        let mut c: Option<C> = None;
        let mut ok = true;
        for i in 0..2usize {
            for j in 0..2usize {
                let a = x[i][j];
                let b = g[i][j];
                if b == *c0 {
                    if a != *c0 { ok = false; }
                } else {
                    let den = b.0.mul(&b.0).add(&b.1.mul(&b.1));
                    let p = a.0.mul(&b.0).add(&a.1.mul(&b.1)).div(&den);
                    let q = a.1.mul(&b.0).sub(&a.0.mul(&b.1)).div(&den);
                    match c {
                        None => c = Some(C(p, q)),
                        Some(cc) => if cmul(cc, b) != a { ok = false; },
                    }
                }
            }
        }
        if ok { if let Some(cc) = c { return Some((k, cc)); } }
    }
    None
}

fn coeff_ad(t: &Vec<MatC>, k: usize, j: usize) -> Vec<i64> {
    let c = crochet(&t[k], &t[j]);
    let mut col = vec![0i64, 0, 0];
    for i in 0..3usize {
        if meq(&c, &t[i]) { col[i] = 1; }
        else if meq(&c, &neg_mat(&t[i])) { col[i] = -1; }
    }
    col
}

fn chi_a1(_g: &Mat3) -> i64 { 1 }
fn chi_a2(g: &Mat3) -> i64 { g[2][2] }
fn chi_e(g: &Mat3) -> i64 { g[0][0] + g[1][1] }
fn chi_dir(g: &Mat3) -> i64 { g[0][0] + g[1][1] + g[2][2] }
fn chi_pair(g: &Mat3) -> i64 { 1 + chi_dir(g) }

fn scalaire_interne(axe: &Vec<Mat3>, f: fn(&Mat3) -> i64, h: fn(&Mat3) -> i64) -> Fract {
    let somme: i128 = axe.iter().map(|g| (f(g) * h(g)) as i128).sum();
    Fract::new(somme, axe.len() as i128)
}

fn commutant(mats: &Vec<MatN>, n: usize) -> usize {
    let mut lignes: Vec<Vec<Fract>> = Vec::new();
    for g in mats.iter() {
        for i in 0..n {
            for j in 0..n {
                let mut ligne = vec![Fract::zero(); n * n];
                for k in 0..n {
                    ligne[n * i + k] = ligne[n * i + k].add(&Fract::new(g[k][j] as i128, 1));
                    ligne[n * k + j] = ligne[n * k + j].sub(&Fract::new(g[i][k] as i128, 1));
                }
                lignes.push(ligne);
            }
        }
    }
    let mut piv = 0usize;
    for col in 0..(n * n) {
        let sel = (piv..lignes.len()).find(|&r| !lignes[r][col].is_zero());
        let sel = match sel { Some(s) => s, None => continue };
        lignes.swap(piv, sel);
        let pv = lignes[piv][col];
        for v in lignes[piv].iter_mut() { *v = v.div(&pv); }
        for r in 0..lignes.len() {
            if r != piv && !lignes[r][col].is_zero() {
                let f = lignes[r][col];
                for c in 0..(n * n) {
                    lignes[r][c] = lignes[r][c].sub(&f.mul(&lignes[piv][c]));
                }
            }
        }
        piv += 1;
    }
    n * n - piv
}

// Un autre point rationnel : le point pythagoricien (5/13, 12/13).


    // TODO etudiant : construire R = [[c, -s, 0], [s, c, 0], [0, 0, 1]],
    // verifier R^T R = I et det R = 1 en Fraction exactes
    let _c = Fract::new(5, 13);
    let _s = Fract::new(12, 13);
    let _resultat: Option<Vec<Vec<Fract>>> = None;
    println!("Exercice a completer");


Exercice a completer


### Exercice 3 — le commutant du groupe entier

Le commutant du stabilisateur d'axe (ordre 8) sur les directions vaut deux, sur l'espace de paire trois. Que vaut le commutant du groupe **entier** des 24 rotations sur $\mathbb{R}^3$ ? Mesurer avec la même fonction `commutant`.

In [11]:
// Bloc d'outillage commun : identique dans chaque cellule de code du carnet.
// evcxr conserve les items definis d'une cellule a l'autre ; un item de MEME nom
// redefini avec un contenu different casse les cellules voisines. On garde donc
// un prefixe canonique, repete mot pour mot, et chaque cellule n'ajoute que
// ses helpers propres.
use std::collections::BTreeMap;

#[derive(Clone, Copy, PartialEq, Eq, Debug)]
struct Fract { num: i128, den: i128 }

fn gcd(mut a: i128, mut b: i128) -> i128 {
    while b != 0 { let t = b; b = a % b; a = t; }
    a
}

impl Fract {
    fn new(num: i128, den: i128) -> Self {
        assert!(den != 0, "denominateur nul");
        let mut g = gcd(num.abs(), den.abs());
        if g == 0 { g = 1; }
        let mut n = num / g;
        let mut d = den / g;
        if d < 0 { n = -n; d = -d; }
        Fract { num: n, den: d }
    }
    fn zero() -> Self { Fract { num: 0, den: 1 } }
    fn add(&self, o: &Fract) -> Fract { Fract::new(self.num * o.den + o.num * self.den, self.den * o.den) }
    fn sub(&self, o: &Fract) -> Fract { Fract::new(self.num * o.den - o.num * self.den, self.den * o.den) }
    fn mul(&self, o: &Fract) -> Fract { Fract::new(self.num * o.num, self.den * o.den) }
    fn div(&self, o: &Fract) -> Fract { Fract::new(self.num * o.den, self.den * o.num) }
    fn neg(&self) -> Fract { Fract { num: -self.num, den: self.den } }
    fn is_zero(&self) -> bool { self.num == 0 }
}

impl std::fmt::Display for Fract {
    fn fmt(&self, f: &mut std::fmt::Formatter<'_>) -> std::fmt::Result {
        if self.den == 1 { write!(f, "{}", self.num) } else { write!(f, "{}/{}", self.num, self.den) }
    }
}

#[derive(Clone, Copy, PartialEq, Eq, Debug)]
struct C(Fract, Fract);

fn cmul(x: C, y: C) -> C { C(x.0.mul(&y.0).sub(&x.1.mul(&y.1)), x.0.mul(&y.1).add(&x.1.mul(&y.0))) }
fn cadd(x: C, y: C) -> C { C(x.0.add(&y.0), x.1.add(&y.1)) }
fn cneg(x: C) -> C { C(x.0.neg(), x.1.neg()) }
fn cscale(x: C, n: &Fract) -> C { C(x.0.mul(n), x.1.mul(n)) }

type MatC = Vec<Vec<C>>;
type Mat3 = [[i64; 3]; 3];
type MatN = Vec<Vec<i64>>;

fn mmul(a: &MatC, b: &MatC) -> MatC {
    let mut out = Vec::new();
    for i in 0..a.len() {
        let mut row = Vec::new();
        for j in 0..b[0].len() {
            let mut acc = C(Fract::zero(), Fract::zero());
            for k in 0..b.len() { acc = cadd(acc, cmul(a[i][k], b[k][j])); }
            row.push(acc);
        }
        out.push(row);
    }
    out
}

fn msub(a: &MatC, b: &MatC) -> MatC {
    let mut out = Vec::new();
    for i in 0..a.len() {
        let mut row = Vec::new();
        for j in 0..a[0].len() { row.push(cadd(a[i][j], cneg(b[i][j]))); }
        out.push(row);
    }
    out
}

fn crochet(a: &MatC, b: &MatC) -> MatC { msub(&mmul(a, b), &mmul(b, a)) }

fn meq(a: &MatC, b: &MatC) -> bool {
    (0..a.len()).all(|i| (0..a[0].len()).all(|j| a[i][j] == b[i][j]))
}

fn neg_mat(m: &MatC) -> MatC {
    m.iter().map(|row| row.iter().map(|x| cneg(*x)).collect()).collect()
}

fn mscale2(m: &MatC, k: i64) -> MatC {
    m.iter().map(|row| row.iter().map(|x| cscale(*x, &Fract::new(k as i128, 1))).collect()).collect()
}

fn det3(m: &Mat3) -> i64 {
    m[0][0] * (m[1][1] * m[2][2] - m[1][2] * m[2][1])
        - m[0][1] * (m[1][0] * m[2][2] - m[1][2] * m[2][0])
        + m[0][2] * (m[1][0] * m[2][1] - m[1][1] * m[2][0])
}

fn mul3(a: &Mat3, b: &Mat3) -> Mat3 {
    let mut out = [[0i64; 3]; 3];
    for i in 0..3 {
        for j in 0..3 {
            let mut acc = 0i64;
            for k in 0..3 { acc += a[i][k] * b[k][j]; }
            out[i][j] = acc;
        }
    }
    out
}

const I3: Mat3 = [[1, 0, 0], [0, 1, 0], [0, 0, 1]];

fn ordre3(m: &Mat3) -> i64 {
    let mut r = *m;
    let mut k = 1i64;
    while r != I3 { r = mul3(&r, m); k += 1; }
    k
}

fn py_bool(b: bool) -> String { if b { "True".to_string() } else { "False".to_string() } }

fn py_bools(items: &[bool]) -> String {
    format!("[{}]", items.iter().map(|b| py_bool(*b)).collect::<Vec<String>>().join(", "))
}

fn proportionnel(x: &MatC, base: &[MatC], c0: &C) -> Option<(usize, C)> {
    for (k, g) in base.iter().enumerate() {
        let mut c: Option<C> = None;
        let mut ok = true;
        for i in 0..2usize {
            for j in 0..2usize {
                let a = x[i][j];
                let b = g[i][j];
                if b == *c0 {
                    if a != *c0 { ok = false; }
                } else {
                    let den = b.0.mul(&b.0).add(&b.1.mul(&b.1));
                    let p = a.0.mul(&b.0).add(&a.1.mul(&b.1)).div(&den);
                    let q = a.1.mul(&b.0).sub(&a.0.mul(&b.1)).div(&den);
                    match c {
                        None => c = Some(C(p, q)),
                        Some(cc) => if cmul(cc, b) != a { ok = false; },
                    }
                }
            }
        }
        if ok { if let Some(cc) = c { return Some((k, cc)); } }
    }
    None
}

fn coeff_ad(t: &Vec<MatC>, k: usize, j: usize) -> Vec<i64> {
    let c = crochet(&t[k], &t[j]);
    let mut col = vec![0i64, 0, 0];
    for i in 0..3usize {
        if meq(&c, &t[i]) { col[i] = 1; }
        else if meq(&c, &neg_mat(&t[i])) { col[i] = -1; }
    }
    col
}

fn chi_a1(_g: &Mat3) -> i64 { 1 }
fn chi_a2(g: &Mat3) -> i64 { g[2][2] }
fn chi_e(g: &Mat3) -> i64 { g[0][0] + g[1][1] }
fn chi_dir(g: &Mat3) -> i64 { g[0][0] + g[1][1] + g[2][2] }
fn chi_pair(g: &Mat3) -> i64 { 1 + chi_dir(g) }

fn scalaire_interne(axe: &Vec<Mat3>, f: fn(&Mat3) -> i64, h: fn(&Mat3) -> i64) -> Fract {
    let somme: i128 = axe.iter().map(|g| (f(g) * h(g)) as i128).sum();
    Fract::new(somme, axe.len() as i128)
}

fn commutant(mats: &Vec<MatN>, n: usize) -> usize {
    let mut lignes: Vec<Vec<Fract>> = Vec::new();
    for g in mats.iter() {
        for i in 0..n {
            for j in 0..n {
                let mut ligne = vec![Fract::zero(); n * n];
                for k in 0..n {
                    ligne[n * i + k] = ligne[n * i + k].add(&Fract::new(g[k][j] as i128, 1));
                    ligne[n * k + j] = ligne[n * k + j].sub(&Fract::new(g[i][k] as i128, 1));
                }
                lignes.push(ligne);
            }
        }
    }
    let mut piv = 0usize;
    for col in 0..(n * n) {
        let sel = (piv..lignes.len()).find(|&r| !lignes[r][col].is_zero());
        let sel = match sel { Some(s) => s, None => continue };
        lignes.swap(piv, sel);
        let pv = lignes[piv][col];
        for v in lignes[piv].iter_mut() { *v = v.div(&pv); }
        for r in 0..lignes.len() {
            if r != piv && !lignes[r][col].is_zero() {
                let f = lignes[r][col];
                for c in 0..(n * n) {
                    lignes[r][c] = lignes[r][c].sub(&f.mul(&lignes[piv][c]));
                }
            }
        }
        piv += 1;
    }
    n * n - piv
}

// Le commutant du groupe entier des 24 rotations.


    // TODO etudiant : commutant(rotations24, 3) et interpreter (la representation
    // naturelle des rotations du cube est irreductible -- attendu : 1)
    let _compte: Option<usize> = None;
    println!("Exercice a completer");


Exercice a completer


## Conclusion

Le groupe des rotations n'est pas posé, il est dérivé : trois directions, leurs matrices *ad*, une algèbre de dimension trois aux crochets du symbole alterné — et un double recouvrement mesuré ($-I$ au bout qubit, $+I$ sur les directions). L'image finie est le diédral d'ordre huit à indice trois dans les vingt-quatre rotations du réseau, et le census typé des 102 bouts reste cité, non rejoué. « The rotations fixed, the next chapter joins them to the order coordinate in the Lorentz group » — le chapitre suivant soude rotations et coordonnée d'ordre dans le groupe de Lorentz.

In [12]:
// Bloc d'outillage commun : identique dans chaque cellule de code du carnet.
// evcxr conserve les items definis d'une cellule a l'autre ; un item de MEME nom
// redefini avec un contenu different casse les cellules voisines. On garde donc
// un prefixe canonique, repete mot pour mot, et chaque cellule n'ajoute que
// ses helpers propres.
use std::collections::BTreeMap;

#[derive(Clone, Copy, PartialEq, Eq, Debug)]
struct Fract { num: i128, den: i128 }

fn gcd(mut a: i128, mut b: i128) -> i128 {
    while b != 0 { let t = b; b = a % b; a = t; }
    a
}

impl Fract {
    fn new(num: i128, den: i128) -> Self {
        assert!(den != 0, "denominateur nul");
        let mut g = gcd(num.abs(), den.abs());
        if g == 0 { g = 1; }
        let mut n = num / g;
        let mut d = den / g;
        if d < 0 { n = -n; d = -d; }
        Fract { num: n, den: d }
    }
    fn zero() -> Self { Fract { num: 0, den: 1 } }
    fn add(&self, o: &Fract) -> Fract { Fract::new(self.num * o.den + o.num * self.den, self.den * o.den) }
    fn sub(&self, o: &Fract) -> Fract { Fract::new(self.num * o.den - o.num * self.den, self.den * o.den) }
    fn mul(&self, o: &Fract) -> Fract { Fract::new(self.num * o.num, self.den * o.den) }
    fn div(&self, o: &Fract) -> Fract { Fract::new(self.num * o.den, self.den * o.num) }
    fn neg(&self) -> Fract { Fract { num: -self.num, den: self.den } }
    fn is_zero(&self) -> bool { self.num == 0 }
}

impl std::fmt::Display for Fract {
    fn fmt(&self, f: &mut std::fmt::Formatter<'_>) -> std::fmt::Result {
        if self.den == 1 { write!(f, "{}", self.num) } else { write!(f, "{}/{}", self.num, self.den) }
    }
}

#[derive(Clone, Copy, PartialEq, Eq, Debug)]
struct C(Fract, Fract);

fn cmul(x: C, y: C) -> C { C(x.0.mul(&y.0).sub(&x.1.mul(&y.1)), x.0.mul(&y.1).add(&x.1.mul(&y.0))) }
fn cadd(x: C, y: C) -> C { C(x.0.add(&y.0), x.1.add(&y.1)) }
fn cneg(x: C) -> C { C(x.0.neg(), x.1.neg()) }
fn cscale(x: C, n: &Fract) -> C { C(x.0.mul(n), x.1.mul(n)) }

type MatC = Vec<Vec<C>>;
type Mat3 = [[i64; 3]; 3];
type MatN = Vec<Vec<i64>>;

fn mmul(a: &MatC, b: &MatC) -> MatC {
    let mut out = Vec::new();
    for i in 0..a.len() {
        let mut row = Vec::new();
        for j in 0..b[0].len() {
            let mut acc = C(Fract::zero(), Fract::zero());
            for k in 0..b.len() { acc = cadd(acc, cmul(a[i][k], b[k][j])); }
            row.push(acc);
        }
        out.push(row);
    }
    out
}

fn msub(a: &MatC, b: &MatC) -> MatC {
    let mut out = Vec::new();
    for i in 0..a.len() {
        let mut row = Vec::new();
        for j in 0..a[0].len() { row.push(cadd(a[i][j], cneg(b[i][j]))); }
        out.push(row);
    }
    out
}

fn crochet(a: &MatC, b: &MatC) -> MatC { msub(&mmul(a, b), &mmul(b, a)) }

fn meq(a: &MatC, b: &MatC) -> bool {
    (0..a.len()).all(|i| (0..a[0].len()).all(|j| a[i][j] == b[i][j]))
}

fn neg_mat(m: &MatC) -> MatC {
    m.iter().map(|row| row.iter().map(|x| cneg(*x)).collect()).collect()
}

fn mscale2(m: &MatC, k: i64) -> MatC {
    m.iter().map(|row| row.iter().map(|x| cscale(*x, &Fract::new(k as i128, 1))).collect()).collect()
}

fn det3(m: &Mat3) -> i64 {
    m[0][0] * (m[1][1] * m[2][2] - m[1][2] * m[2][1])
        - m[0][1] * (m[1][0] * m[2][2] - m[1][2] * m[2][0])
        + m[0][2] * (m[1][0] * m[2][1] - m[1][1] * m[2][0])
}

fn mul3(a: &Mat3, b: &Mat3) -> Mat3 {
    let mut out = [[0i64; 3]; 3];
    for i in 0..3 {
        for j in 0..3 {
            let mut acc = 0i64;
            for k in 0..3 { acc += a[i][k] * b[k][j]; }
            out[i][j] = acc;
        }
    }
    out
}

const I3: Mat3 = [[1, 0, 0], [0, 1, 0], [0, 0, 1]];

fn ordre3(m: &Mat3) -> i64 {
    let mut r = *m;
    let mut k = 1i64;
    while r != I3 { r = mul3(&r, m); k += 1; }
    k
}

fn py_bool(b: bool) -> String { if b { "True".to_string() } else { "False".to_string() } }

fn py_bools(items: &[bool]) -> String {
    format!("[{}]", items.iter().map(|b| py_bool(*b)).collect::<Vec<String>>().join(", "))
}

fn proportionnel(x: &MatC, base: &[MatC], c0: &C) -> Option<(usize, C)> {
    for (k, g) in base.iter().enumerate() {
        let mut c: Option<C> = None;
        let mut ok = true;
        for i in 0..2usize {
            for j in 0..2usize {
                let a = x[i][j];
                let b = g[i][j];
                if b == *c0 {
                    if a != *c0 { ok = false; }
                } else {
                    let den = b.0.mul(&b.0).add(&b.1.mul(&b.1));
                    let p = a.0.mul(&b.0).add(&a.1.mul(&b.1)).div(&den);
                    let q = a.1.mul(&b.0).sub(&a.0.mul(&b.1)).div(&den);
                    match c {
                        None => c = Some(C(p, q)),
                        Some(cc) => if cmul(cc, b) != a { ok = false; },
                    }
                }
            }
        }
        if ok { if let Some(cc) = c { return Some((k, cc)); } }
    }
    None
}

fn coeff_ad(t: &Vec<MatC>, k: usize, j: usize) -> Vec<i64> {
    let c = crochet(&t[k], &t[j]);
    let mut col = vec![0i64, 0, 0];
    for i in 0..3usize {
        if meq(&c, &t[i]) { col[i] = 1; }
        else if meq(&c, &neg_mat(&t[i])) { col[i] = -1; }
    }
    col
}

fn chi_a1(_g: &Mat3) -> i64 { 1 }
fn chi_a2(g: &Mat3) -> i64 { g[2][2] }
fn chi_e(g: &Mat3) -> i64 { g[0][0] + g[1][1] }
fn chi_dir(g: &Mat3) -> i64 { g[0][0] + g[1][1] + g[2][2] }
fn chi_pair(g: &Mat3) -> i64 { 1 + chi_dir(g) }

fn scalaire_interne(axe: &Vec<Mat3>, f: fn(&Mat3) -> i64, h: fn(&Mat3) -> i64) -> Fract {
    let somme: i128 = axe.iter().map(|g| (f(g) * h(g)) as i128).sum();
    Fract::new(somme, axe.len() as i128)
}

fn commutant(mats: &Vec<MatN>, n: usize) -> usize {
    let mut lignes: Vec<Vec<Fract>> = Vec::new();
    for g in mats.iter() {
        for i in 0..n {
            for j in 0..n {
                let mut ligne = vec![Fract::zero(); n * n];
                for k in 0..n {
                    ligne[n * i + k] = ligne[n * i + k].add(&Fract::new(g[k][j] as i128, 1));
                    ligne[n * k + j] = ligne[n * k + j].sub(&Fract::new(g[i][k] as i128, 1));
                }
                lignes.push(ligne);
            }
        }
    }
    let mut piv = 0usize;
    for col in 0..(n * n) {
        let sel = (piv..lignes.len()).find(|&r| !lignes[r][col].is_zero());
        let sel = match sel { Some(s) => s, None => continue };
        lignes.swap(piv, sel);
        let pv = lignes[piv][col];
        for v in lignes[piv].iter_mut() { *v = v.div(&pv); }
        for r in 0..lignes.len() {
            if r != piv && !lignes[r][col].is_zero() {
                let f = lignes[r][col];
                for c in 0..(n * n) {
                    lignes[r][c] = lignes[r][c].sub(&f.mul(&lignes[piv][c]));
                }
            }
        }
        piv += 1;
    }
    n * n - piv
}

// Releve final : les checks du carnet.


    let C0 = C(Fract::zero(), Fract::zero());
    let C1 = C(Fract::new(1, 1), Fract::zero());
    let CI = C(Fract::zero(), Fract::new(1, 1));

    // --- Les 32 matrices monomiales et le noyau scalaire (cellule 3).
    let racines = [C1, CI, cneg(C1), cneg(CI)];
    let mut monomiales: Vec<MatC> = Vec::new();
    for c0 in 0..2usize {
        for p1 in racines.iter() {
            for p2 in racines.iter() {
                let mut m = vec![vec![C0, C0], vec![C0, C0]];
                m[0][c0] = *p1;
                m[1][1 - c0] = *p2;
                monomiales.push(m);
            }
        }
    }
    let n_scalaires = monomiales.iter()
        .filter(|m| m[0][1] == C0 && m[1][0] == C0 && m[0][0] == m[1][1])
        .count();

    // --- Les 24 rotations et le stabilisateur d'axe (cellule 6).
    let perms: [[usize; 3]; 6] = [
        [0, 1, 2], [0, 2, 1], [1, 0, 2], [1, 2, 0], [2, 0, 1], [2, 1, 0],
    ];
    let signes: [[i64; 3]; 8] = [
        [1, 1, 1], [1, 1, -1], [1, -1, 1], [1, -1, -1],
        [-1, 1, 1], [-1, 1, -1], [-1, -1, 1], [-1, -1, -1],
    ];
    let mut rotations24: Vec<Mat3> = Vec::new();
    for perm in perms.iter() {
        for sg in signes.iter() {
            let mut m = [[0i64; 3]; 3];
            for i in 0..3 { m[i][perm[i]] = sg[i]; }
            if det3(&m) == 1 { rotations24.push(m); }
        }
    }
    let axe: Vec<Mat3> = rotations24.iter()
        .filter(|m| m[0][2] == 0 && m[1][2] == 0 && m[2][0] == 0 && m[2][1] == 0)
        .cloned()
        .collect();
    let mut profil: Vec<i64> = axe.iter().map(ordre3).collect();
    profil.sort_unstable();
    let ferme = axe.iter().all(|a| axe.iter().all(|b| axe.contains(&mul3(a, b))));

    // --- Les cinq classes de conjugaison (cellule 7).
    let mut classes: BTreeMap<(i64, i64, bool), usize> = BTreeMap::new();
    for g in axe.iter() {
        let tr = g[0][0] + g[1][1];
        let hors_diag = g[0][1] != 0;
        *classes.entry((ordre3(g), tr, hors_diag)).or_insert(0) += 1;
    }

    // --- Le triple litteral et sa re-orientation (cellule 10).
    let demi = Fract::new(1, 2);
    let g1 = vec![vec![C0, cscale(cneg(CI), &demi)], vec![cscale(cneg(CI), &demi), C0]];
    let g2 = vec![vec![C0, C(demi, Fract::zero())], vec![cneg(C(demi, Fract::zero())), C0]];
    let g3 = vec![vec![cscale(cneg(CI), &demi), C0], vec![C0, cscale(CI, &demi)]];
    let litteral: Vec<MatC> = vec![g1, g2, g3];
    let mut constantes: Vec<Fract> = Vec::new();
    for (a, b) in [(0usize, 1usize), (1, 2), (2, 0)] {
        let cro = crochet(&litteral[a], &litteral[b]);
        let (k, c) = match proportionnel(&cro, &litteral, &C0) {
            Some(v) => v,
            None => { println!("constante introuvable"); (0, C0) }
        };
        constantes.push(c.0);
        let _ = k;
    }
    let litteral_ok = constantes.iter().all(|c| *c == Fract::new(-1, 1));
    let t: Vec<MatC> = vec![litteral[0].clone(), neg_mat(&litteral[1]), litteral[2].clone()];
    let attendu: [(usize, usize, usize); 3] = [(0, 1, 2), (1, 2, 0), (2, 0, 1)];
    let epsilon_exact = attendu.iter().all(|&(a, b, k)| meq(&crochet(&t[a], &t[b]), &t[k]));
    let moins_quart_i = vec![vec![C(Fract::new(-1, 4), Fract::zero()), C0],
                             vec![C0, C(Fract::new(-1, 4), Fract::zero())]];
    let carres: Vec<bool> = t.iter().map(|x| meq(&mmul(x, x), &moins_quart_i)).collect();

    // --- Le double recouvrement et l'action adjointe (cellule 12).
    let demi_tour: MatC = t[2].iter()
        .map(|row| row.iter().map(|x| cscale(*x, &Fract::new(2, 1))).collect())
        .collect();
    let mut attendu_demi: MatC = vec![vec![C(Fract::zero(), Fract::new(-1, 1)), C0],
                                      vec![C0, C(Fract::zero(), Fract::new(-1, 1))]];
    attendu_demi[1][1] = C(Fract::zero(), Fract::new(1, 1));
    let demi_ok = meq(&demi_tour, &attendu_demi);
    let moins_i: MatC = vec![vec![cneg(C1), C0], vec![C0, cneg(C1)]];
    let exp2pi: Vec<bool> = t.iter().map(|x| meq(&mmul(&mscale2(x, 2), &mscale2(x, 2)), &moins_i)).collect();
    let cols: Vec<Vec<i64>> = (0..3).map(|j| coeff_ad(&t, 2, j)).collect();
    let mut ad_t3: Mat3 = [[0i64; 3]; 3];
    for i in 0..3 {
        for j in 0..3 { ad_t3[i][j] = cols[j][i]; }
    }
    let j2 = mul3(&ad_t3, &ad_t3);
    let mut quart: Mat3 = [[0i64; 3]; 3];
    for i in 0..3 {
        for j in 0..3 { quart[i][j] = I3[i][j] + ad_t3[i][j] + j2[i][j]; }
    }
    let q2 = mul3(&quart, &quart);
    let q4 = mul3(&q2, &q2);
    let ad_tour_complet = q4 == I3;
    let double_couvre = demi_ok && exp2pi.iter().all(|b| *b) && ad_tour_complet;

    // --- Les caracteres et les commutants (cellule 15).
    let mut axe_n: Vec<MatN> = Vec::new();
    for g in axe.iter() {
        axe_n.push((0..3).map(|i| (0..3).map(|j| g[i][j]).collect()).collect());
    }
    let mut pair4: Vec<MatN> = Vec::new();
    for g in axe.iter() {
        let mut b = vec![vec![0i64; 4]; 4];
        b[0][0] = 1;
        for i in 0..3 {
            for j in 0..3 { b[1 + i][1 + j] = g[i][j]; }
        }
        pair4.push(b);
    }
    let d_pair = commutant(&pair4, 4);

    // --- Le releve final.
    let checks: Vec<(&str, bool)> = vec![
        ("32 monomiales, 4 scalaires, 32/4 = 8",
         monomiales.len() == 32 && n_scalaires == 4
         && Fract::new(monomiales.len() as i128, n_scalaires as i128) == Fract::new(8, 1)),
        ("24 rotations du reseau", rotations24.len() == 24),
        ("stabilisateur d'axe = 8, ferme, profil D8",
         axe.len() == 8 && ferme && profil == vec![1, 2, 2, 2, 2, 2, 4, 4]),
        ("cinq classes (A1, A2, B1, B2, E)", classes.len() == 5),
        ("crochets littéraux = (-1, -1, -1)", litteral_ok),
        ("re-orientation -> eps_abl exact", epsilon_exact),
        ("Tj^2 = -(1/4)I", carres.iter().all(|b| *b)),
        ("exp(2*pi*Tj) = -I au bout, +I sur directions", double_couvre),
        ("chi_dir = A2 + E, chi_pair = A1 + A2 + E",
         scalaire_interne(&axe, chi_dir, chi_a2) == Fract::new(1, 1)
         && scalaire_interne(&axe, chi_dir, chi_e) == Fract::new(1, 1)
         && scalaire_interne(&axe, chi_pair, chi_a1) == Fract::new(1, 1)),
        ("quart de tour = exp(pi/2 ad_T3), puissance 4 = I3",
         quart == [[0, -1, 0], [1, 0, 0], [0, 0, 1]] && ad_tour_complet),
        ("commutant espace de paire = 3", d_pair == 3),
    ];
    let ok = checks.iter().filter(|(_, v)| *v).count();
    for (k, v) in checks.iter() {
        println!("{} -> {}", k, py_bool(*v));
    }
    println!("Total : {} / {}", ok, checks.len());


32 monomiales, 4 scalaires, 32/4 = 8 -> True


24 rotations du reseau -> True


stabilisateur d'axe = 8, ferme, profil D8 -> True


cinq classes (A1, A2, B1, B2, E) -> True


crochets littéraux = (-1, -1, -1) -> True


re-orientation -> eps_abl exact -> True


... (2 outputs left out)

chi_dir = A2 + E, chi_pair = A1 + A2 + E -> True


quart de tour = exp(pi/2 ad_T3), puissance 4 = I3 -> True


commutant espace de paire = 3 -> True


Total : 11 / 11


## À retenir

1. **Le groupe n'est pas posé, il est dérivé** : trois directions, leurs matrices *ad*, une
   algèbre de dimension trois fermée par les crochets du symbole alterné — et $T_j^2 = -\tfrac14 I$
   aux trois axes, mesuré.
2. **Les comptes se referment** : $32$ matrices monomiales à phases racines quatrièmes, exactement
   $4$ scalaires, image d'ordre $32/4 = 8$ par Lagrange ; et $24$ rotations du réseau
   (permutations signées de déterminant $+1$), dont le stabilisateur d'axe est un sous-groupe
   d'indice $3$, fermé par produit, profil par ordres $[1,2,2,2,2,2,4,4]$ — $D_8$, pas $Q_8$.
3. **Cinq classes pour cinq irréductibles** : le tri (ordre, trace du bloc $xy$, diagonale)
   distingue exactement cinq classes, et les caractères se décomposent : directions
   $= A_2 \oplus E$, espace de paire $= A_1 \oplus A_2 \oplus E$, commutants $2$ et $3$.
4. **Le quart de tour adjoint se calcule** : $\mathrm{ad}_{T_3} = J$ vérifie $J^3 = -J$, donc
   $\exp(\tfrac{\pi}{2} J) = I + J + J^2$ — la forme fermée donne exactement la rotation
   planaire, et sa puissance quatrième vaut $I_3$ **sur les directions**.
5. **Le double recouvrement est mesuré, pas cité** : $(2T_j)^2 = 4T_j^2 = -I$ au bout qubit pour
   les trois axes, pendant que le tour complet agit trivialement sur les directions — c'est
   l'écart $\pm I$ qui distingue les deux bouts.


## Vérifiez votre compréhension

1. Pourquoi $32/4 = 8$ ne prouve-t-il pas que l'image est diédrale, et que faut-il de plus pour
   l'établir ?
2. Comment le profil par ordres distingue-t-il $D_8$ de $Q_8$, et que lit-on sur ses cinq classes
   de conjugaison ?
3. Que signifie « le triple littéral à constantes de structure $(-1,-1,-1)$ » et qu'est-ce qu'une
   ré-orientation de signe y change ?
4. Pourquoi $\exp(\tfrac{\pi}{2}\,\mathrm{ad}_{T_3})$ vaut-il $I + J + J^2$, et que mesure
   la puissance quatrième de cette matrice ?

<details>
<summary>Réponses</summary>

1. L'arithmétique $32 = 4 \times 8$ est une **cohérence** de Lagrange : elle contraint l'indice du
   noyau, elle ne dit rien de la structure de l'image. Ce qui l'établit, c'est la mesure directe —
   le stabilisateur d'axe est fermé par produit et son profil par ordres est $[1,2,2,2,2,2,4,4]$,
   signature de $D_8$ ($Q_8$ aurait six éléments d'ordre $4$).
2. $D_8$ a cinq éléments d'ordre $2$ et deux d'ordre $4$ ; $Q_8$ aurait six éléments d'ordre $4$
   et un seul d'ordre $2$. Les cinq classes de conjugaison (neutre, demi-tour central, deux quarts
   de tour, réflexions diagonales, réflexions hors-diagonales) correspondent aux cinq irréductibles
   $A_1, A_2, B_1, B_2, E$.
3. C'est l'algèbre de Lie engendrée par les trois générateurs du chapitre 55, dont les crochets
   littéraux donnent les constantes $(-1,-1,-1)$ ; une **seule** ré-orientation de signe sur les
   générateurs les ramène au symbole alterné $\epsilon_{abc}$ exact — l'algèbre est $\mathfrak{so}(3)$,
   écrit dans une base mal orientée.
4. Parce que $\mathrm{ad}_{T_3}$ a la structure $J$ avec $J^3 = -J$ : la série entière se referme en
   $I + \sin\theta\,J + (1-\cos\theta)J^2$, et au quart de tour $\sin = 1$, $\cos = 0$, d'où
   $I + J + J^2$. Sa puissance quatrième vaut $I_3$ : le quart de tour est d'ordre $4$ sur les
   directions, alors que le tour complet vaut $-I$ au bout qubit.

</details>


## Pour aller plus loin

*Chaînes-11 — le groupe de Lorentz et sa signature*
est le carnet que la conclusion annonce : « the next chapter joins them to the order coordinate in
the Lorentz group » — les rotations dérivées ici y rencontrent la coordonnée d'ordre du chapitre 52.

En amont, [Chaînes-09 — la coordonnée d'ordre](Chaines-09-Coordonnee-Ordre-Chaine-Serie-Python.ipynb)
est le carnet dont la conclusion annonçait celui-ci (« with order read as a count, the next chapter
turns to the rotations »), et [Chaînes-08 — les corollaires à leurs racines](Chaines-08-Corollaires-Quantique-Racines-Python.ipynb)
en fournit les corollaires quantiques.

Hors de la série, [Jauge-01 — le théorème de Noether](../Jauge/Jauge-01-Theoreme-Noether-Python.ipynb)
reste le contraste de méthode : là où une conservation se dérive d'une symétrie continue, le groupe
décrit ici est **fini**, et c'est son action dérivée — non postulée — qui fait le théorème 53.1.

> Carnet issu de l'EPIC #26510.
